# 05 Time heterogeneity models

This notebook reuses the 04 model pipeline, splits the sample into six four-hour local-time windows, removes time-window predictors from X, runs one OLS model per window, and exports side-by-side coefficient comparison tables.


In [1]:
# =========================================================
# 0. Imports and model control panel
#    Python 3.8 compatible
# =========================================================
import json
import math
import os
import re
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------
# Target
# ---------------------------------------------------------
Y_COL = "speed_kmh"
# Other common targets:
# Y_COL = "duration"
# Y_COL = "final_distance_m"
# Y_COL = "overspeed_20"
# Y_COL = "speed_kmh_minus_service_time"

# raw, log, log1p, sqrt, signed_log1p
# Y_TRANSFORM = "log1p"
Y_TRANSFORM = "raw"
ROW_QUERY = None

# ---------------------------------------------------------
# Model run controls
# ---------------------------------------------------------
# If RUN_MULTIPLE_MODEL_SPECS is True, MODEL_RUN_SPECS controls all row filters.
# Each key becomes a separate output folder under BASE_OUTPUT_ROOT.
RUN_MULTIPLE_MODEL_SPECS = True
PRIMARY_RUN_NAME = "h00_04"

# Six four-hour local-time windows. Each run uses the same covariates, excluding
# time-window indicators, so coefficients are comparable across time blocks.
TIME_HETEROGENEITY_WINDOWS = OrderedDict([
    ("h00_04", {"label": "00:00-04:00", "hour_start": 0, "hour_end": 4}),
    ("h04_08", {"label": "04:00-08:00", "hour_start": 4, "hour_end": 8}),
    ("h08_12", {"label": "08:00-12:00", "hour_start": 8, "hour_end": 12}),
    ("h12_16", {"label": "12:00-16:00", "hour_start": 12, "hour_end": 16}),
    ("h16_20", {"label": "16:00-20:00", "hour_start": 16, "hour_end": 20}),
    ("h20_24", {"label": "20:00-24:00", "hour_start": 20, "hour_end": 24}),
])

BASE_ROW_QUERY_FOR_TIME_HETEROGENEITY = "speed_kmh <= 40"

MODEL_RUN_SPECS = OrderedDict([
    (
        run_name,
        "%s and start_hour_local >= %d and start_hour_local < %d" % (
            BASE_ROW_QUERY_FOR_TIME_HETEROGENEITY,
            info["hour_start"],
            info["hour_end"],
        ),
    )
    for run_name, info in TIME_HETEROGENEITY_WINDOWS.items()
])

# Keep expensive diagnostics focused by default. Turn these on when you need every run fully audited.
RUN_DISTRIBUTION_DASHBOARD_FOR_ALL_SPECS = False
RUN_R2_DIAGNOSTICS = False
RUN_R2_DIAGNOSTICS_FOR_ALL_SPECS = False
RUN_XGBOOST_SHAP_FOR_ALL_SPECS = False

# Optional cluster robust standard errors.
CLUSTER_COL = "courier_id"

# ---------------------------------------------------------
# Variable whitelist
# ---------------------------------------------------------
# Only variables listed here, plus X_EXTRA, can enter the model.
# If a listed variable is absent, it is reported in missing_whitelist_variables.csv.

GRAPHML_ROAD_CLASS_SHARE_COLS = [
    "road_class_share_levelFourRoad_lenw_mean",
    "road_class_share_levelThreeRoad_lenw_mean",
    "road_class_share_secondaryRoad_lenw_mean",
    "road_class_share_nationalRoad_lenw_mean",
    "road_class_share_provincialRoad_lenw_mean",
    # "road_class_share_overPass_lenw_mean",
    # "road_class_share_underPass_lenw_mean",
]

ROAD_TOPOLOGY_GEOMETRY_COLS = [
    "curvature_deg_per_m_lenw_mean",
    "intersection_density_per_km_300m_lenw_mean",
    # "degree_z_mean_end_lenw_mean",
    "betweenness_log1p_z_mean_end_lenw_mean",
    "closeness_per_km_z_mean_end_lenw_mean",
]

POI_LANDUSE_COLS = [
    # "restaurant_shp_count_30m_lenw_mean",
    "poi_count_30m_lenw_mean",
    "poi_mix_entropy_norm_30m_lenw_mean",
    # "poi_count_restaurant_30m_lenw_mean",
]

TIME_WINDOW_COLS = []  # removed from X for time-heterogeneity models

X_GROUPS = OrderedDict([
    ("trip_state", [
        # "duration",
        "onhand_order_count_start",
        "time_pressure_min",
        "is_weekend_local",
        # "final_distance_m",
        # "start_hour_sin",
        # "start_hour_cos",
    ]),
    # Time-window indicators are excluded because each model is run inside a time block.
    ("time_window", TIME_WINDOW_COLS),
    ("rider_behavior", [
        "rider_avg_orders_per_active_day",
        "full_time",
        # "rider_active_day_share_in_data",
        # "rider_median_onhand_raw",
        # "rider_median_max_onhand_per_wave",
        # "rider_share_batch_wave_ge2",
        # "rider_median_segment_distance_m",
        "rider_mean_segment_distance_m",
    ]),
    ("road_class_shares", GRAPHML_ROAD_CLASS_SHARE_COLS),
    ("road_topology_geometry", ROAD_TOPOLOGY_GEOMETRY_COLS),
    ("landuse", POI_LANDUSE_COLS),
])

# Add variables without editing the groups above.
# Derived numeric variables can also enter here once declared in DERIVED_VARIABLE_SPECS.
X_EXTRA = [
    # "time_pressure_urgent",
    # "time_pressure_late",
    # "time_pressure_tight",
]

# ---------------------------------------------------------
# Derived variable controls
# ---------------------------------------------------------
# These are created before the whitelist and categorical controls are applied.
DERIVED_VARIABLE_SPECS = OrderedDict([
    # ("time_pressure_bucket", {
    #     "type": "cut",
    #     "source": "time_pressure_min",
    #     "bins": [-np.inf, 0, 5, 15, 30, np.inf],
    #     "labels": ["lt0", "0_5", "5_15", "15_30", "ge30"],
    #     "categorical": True,
    # }),
    # ("time_pressure_urgent", {
    #     "type": "indicator",
    #     "condition": "time_pressure_min < 5",
    #     "categorical": False,
    # }),
    # ("time_pressure_late", {
    #     "type": "indicator",
    #     "condition": "time_pressure_min < 0",
    #     "categorical": False,
    # }),
    # ("time_pressure_tight", {
    #     "type": "indicator",
    #     "condition": "(time_pressure_min >= 0) & (time_pressure_min < 5)",
    #     "categorical": False,
    # }),

    # -----------------------------------------------------
    # Time-window indicators
    # Baseline group is all other hours, where all three indicators equal 0.
    # Intervals are left-closed and right-open:
    # 10 <= hour < 14, 16 <= hour < 20, 2 <= hour < 6.
    # -----------------------------------------------------
    ("time_midday_peak_10_14", {
        "type": "indicator",
        "condition": "(start_hour_local >= 10) & (start_hour_local < 14)",
        "categorical": False,
    }),
    ("time_evening_peak_16_20", {
        "type": "indicator",
        "condition": "(start_hour_local >= 16) & (start_hour_local < 20)",
        "categorical": False,
    }),
    ("time_midnight_2_6", {
        "type": "indicator",
        "condition": "(start_hour_local >= 2) & (start_hour_local < 6)",
        "categorical": False,
    }),

    # ------------------------------------------------------
    # Rider activity indicators
    # ------------------------------------------------
    ("full_time", {
        "type": "indicator",
        "condition": "(rider_active_day_share_in_data == 1)",
        "categorical": False,
    }),
])

# Remove variables after whitelist selection.
X_DROP = []

# Categorical controls are dummy-encoded with drop_first=True.
# Add columns only when they exist in your model table.
CATEGORICAL_VARS = [
    # Main action-context controls. segment_task_orientation uses parent_end_action
    # to infer the target for FETCH -> GRAB and DELIVER -> GRAB rows.
    "segment_task_orientation",
    # "segment_grab_context",
    # Robustness option:
    # "action_pair",
    # "time_pressure_bucket",
    # "start_hour_local",
]

# ---------------------------------------------------------
# Interaction controls
# ---------------------------------------------------------
# Interactions are created after continuous transforms and before the design matrix.
# Categorical left variables are expanded with drop_first=True, matching build_design_matrix.
ADD_INTERACTIONS = False
INTERACTION_SPECS = [
    # {
    #     "name": "action_pair_x_topology",
    #     "left": ["action_pair"],
    #     "right": ROAD_TOPOLOGY_GEOMETRY_COLS,
    # },
]

# Speed is mechanically distance divided by duration. Turn this on for a more behavioral speed model.
DROP_MECHANICAL_X_WHEN_Y_IS_SPEED = False
MECHANICAL_X_WHEN_Y_SPEED = [
    # "duration",
    # "final_distance_m",
    # "straight_distance_m",
    # "dist_network_only",
]

# Road-class shares sum to one when all classes are included, so one reference class is dropped when an intercept is used.
DROP_ONE_ROAD_CLASS_REFERENCE = True
ROAD_CLASS_REFERENCE_PREFERENCE = ["road_class_share_levelFourRoad_lenw_mean"]
ROAD_CLASS_REFERENCE_MODE = "preference_then_largest"  # preference_then_largest, auto_largest, auto_smallest

# These variables are never allowed in X.
PROTECTED_NON_X_BASE = [
    "overspeed_20",
    "speed_kmh",
    "speed_kmh_minus_service_time",
    "travel_time_minus_service_sec",
    "final_distance_m",
    "path_len_m_from_edges",
    "dist_network_only",
    "start_time",
    "end_time",
    "parent_final_distance_m",
    "parent_duration",
]

# ---------------------------------------------------------
# Transformation controls
# ---------------------------------------------------------
# Transform rules for continuous X variables.
#
# Supported single rules:
#   raw
#   log
#   log1p
#   sqrt
#   signed_log1p
#   logit01
#   standardize
#   div:<number>      example: "div:1000"
#   mul:<number>      example: "mul:60"
#   add:<number>      example: "add:1"
#
# Supported chained rules with "|":
#   "div:1000|log1p"
#   "log1p|standardize"
#
# Put only numeric variables here.
VARIABLE_TRANSFORMS = OrderedDict([
    # ("duration", "log1p"),
    # ("poi_count_restaurant_300m_lenw_mean", "log1p"),
    ("poi_count_30m_lenw_mean", "log1p"),
    # ("speed_kmh", "div:0.7"),
    # ("rider_avg_orders_per_active_day", "log1p"),
    # ("curvature_deg_per_m_lenw_mean", "mul:1000"),
    # ("betweenness_raw_mean_end_lenw_mean", "div:10000|log1p"),
    ("restaurant_shp_count_30m_lenw_mean", "log1p"),    
    ("rider_mean_segment_distance_m", "div:1000"),
    # ("road_len_in_300m_lenw_mean", "log1p"),
    # ("intersection_density_per_km_300m_lenw_mean", "log1p"),
])

PATTERN_VARIABLE_TRANSFORMS = OrderedDict([
    # (r"road_class_", "log1p"),
    # (r"_mean", "log1p"),
    # (r"_count_", "log1p"),
])

AUTO_APPLY_RECOMMENDED_TRANSFORMS = False
RECOMMEND_SKEW_THRESHOLD = 1.0
RECOMMEND_MIN_UNIQUE = 20

# If exact collinearity occurs, columns earlier in this list are kept first.
# This prevents a variable such as restaurant count from being dropped just because an equivalent column appears earlier.
ALIAS_KEEP_PREFERENCE_RAW = [
    "poi_count_restaurant_30m_lenw_mean",
    "restaurant_shp_count_30m_lenw_mean",
]

# VIF controls. VIF is computed with an intercept using auxiliary regressions.
# AUTO_DROP_HIGH_VIF remains off by default to avoid silent model changes.
AUTO_DROP_HIGH_VIF = False
VIF_THRESHOLD = 10.0
VIF_PROTECTED = ["const"]
VIF_MAX_ROWS = 100000
VIF_DRIVER_MAX_ROWS = 100000

# OLS options.
RUN_OLS = True
OLS_STANDARDIZE_X_FOR_ESTIMATION = False
SAVE_DIAGNOSTIC_PLOTS = False
PLOT_SAMPLE_N = 80000

# Distribution dashboard options.
RUN_VARIABLE_DISTRIBUTION_DASHBOARD = False
MAX_DISTRIBUTION_PLOT_VARS = 80
DISTRIBUTION_PLOT_SAMPLE_N = 100000
DISTRIBUTION_HIST_BINS = 60

# XGBoost and SHAP options.
RUN_XGBOOST_SHAP = False
XGB_TEST_SIZE = 0.20
XGB_MAX_ROWS = 250000
SHAP_SAMPLE_N = 20000
SHAP_TOP_N = 20
XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.04,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 1.0,
    "min_child_weight": 5,
    "objective": "reg:squarederror",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}


In [2]:
# =========================================================
# 1. Paths and output structure
# =========================================================
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR.parent,
    Path.cwd(),
    Path("/mnt/data"),
]

PIPELINE_ROOT = BASE_DIR / "outputs_pipeline_aligned"
MANIFEST_PATH = PIPELINE_ROOT / "pipeline_manifest.json"
PIPELINE_MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}
BASE_OUTPUT_ROOT = Path(PIPELINE_MANIFEST.get("time_heterogeneity_root", PIPELINE_ROOT / "05_time_heterogeneity_models"))
OUTPUT_ROOT = BASE_OUTPUT_ROOT
FULL_PRECISION_DIR = OUTPUT_ROOT / "full_precision"
ROUNDED_DIR = OUTPUT_ROOT / "rounded"
DIAGNOSTICS_DIR = OUTPUT_ROOT / "diagnostics"
PLOTS_DIR = OUTPUT_ROOT / "plots"


def set_output_dirs_for_run(run_name=None):
    """Set global output directories.

    Important behavior:
    - With run_name=None, only BASE_OUTPUT_ROOT is created.
    - Per-run folders such as full_precision, rounded, diagnostics, and plots are
      created only inside an actual run folder.
    This avoids empty root-level folders under 05_time_heterogeneity_models.
    """
    global OUTPUT_ROOT, FULL_PRECISION_DIR, ROUNDED_DIR, DIAGNOSTICS_DIR, PLOTS_DIR

    if run_name is None or str(run_name).strip() == "":
        OUTPUT_ROOT = BASE_OUTPUT_ROOT
        BASE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        FULL_PRECISION_DIR = OUTPUT_ROOT / "full_precision"
        ROUNDED_DIR = OUTPUT_ROOT / "rounded"
        DIAGNOSTICS_DIR = OUTPUT_ROOT / "diagnostics"
        PLOTS_DIR = OUTPUT_ROOT / "plots"
        return OUTPUT_ROOT

    safe_run_name = re.sub(r"[^A-Za-z0-9_]+", "_", str(run_name)).strip("_")
    OUTPUT_ROOT = BASE_OUTPUT_ROOT / safe_run_name
    FULL_PRECISION_DIR = OUTPUT_ROOT / "full_precision"
    ROUNDED_DIR = OUTPUT_ROOT / "rounded"
    DIAGNOSTICS_DIR = OUTPUT_ROOT / "diagnostics"
    PLOTS_DIR = OUTPUT_ROOT / "plots"

    for _p in [OUTPUT_ROOT, FULL_PRECISION_DIR, ROUNDED_DIR, DIAGNOSTICS_DIR, PLOTS_DIR]:
        _p.mkdir(parents=True, exist_ok=True)

    return OUTPUT_ROOT

set_output_dirs_for_run(None)

# Clean empty legacy root-level folders that may have been created by older versions.
# Non-empty folders are left untouched.
def cleanup_empty_root_output_subdirs():
    for _name in ["full_precision", "rounded", "diagnostics", "plots"]:
        _p = BASE_OUTPUT_ROOT / _name
        try:
            if _p.exists() and _p.is_dir() and len(list(_p.iterdir())) == 0:
                _p.rmdir()
                print("Removed empty legacy root folder:", _p)
        except Exception as exc:
            print("Could not remove legacy root folder %s: %s" % (_p, exc))

cleanup_empty_root_output_subdirs()

TRIP_TABLE_CANDIDATES = [
    Path(PIPELINE_MANIFEST.get("segment_model_csv_gz", PIPELINE_ROOT / "02_variables" / "segments" / "segment_model_table.csv.gz")),
    PIPELINE_ROOT / "02_variables" / "segments" / "segment_model_table.csv",
]


def resolve_path(candidates, search_dirs=None, required=True):
    if search_dirs is None:
        search_dirs = SEARCH_DIRS
    if isinstance(candidates, (str, Path)):
        candidates = [candidates]
    checked = []
    for cand in candidates:
        p = Path(cand)
        if p.is_absolute():
            checked.append(str(p))
            if p.exists():
                return p
    for root in search_dirs:
        root = Path(root)
        for cand in candidates:
            p = root / cand
            checked.append(str(p))
            if p.exists():
                return p
    if required:
        raise FileNotFoundError("No input found. Checked:\n%s" % "\n".join(checked))
    return Path(candidates[0])

TRIP_TABLE_PATH = resolve_path(TRIP_TABLE_CANDIDATES, required=True)
print("Using model table:", TRIP_TABLE_PATH)
print("Base output root:", BASE_OUTPUT_ROOT)


Using model table: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/02_variables/segments/segment_model_table.csv.gz
Base output root: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models


In [3]:
# =========================================================
# 2. General helper functions
# =========================================================
def ensure_dir(path_like):
    Path(path_like).mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def save_text(text, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")


def read_table(path):
    path = Path(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)


def write_csv_atomic(df, path, **kwargs):
    path = Path(path)
    ensure_dir(path.parent)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, **kwargs)
    os.replace(str(tmp), str(path))


def round_numeric_columns(df, digits=3):
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_bool_dtype(s):
            continue
        if pd.api.types.is_numeric_dtype(s):
            out[col] = pd.to_numeric(s, errors="coerce").round(digits)
    return out


def write_result_table(df, relative_path, index=False, rounded=True, digits=3):
    """Save full precision and optional rounded copy in separated folders."""
    rel = Path(relative_path)
    full_path = FULL_PRECISION_DIR / rel
    write_csv_atomic(df, full_path, index=index)
    if rounded:
        rounded_path = ROUNDED_DIR / rel
        write_csv_atomic(round_numeric_columns(df, digits=digits), rounded_path, index=index)
    return full_path


def make_unique_column_names(cols):
    seen = {}
    out = []
    for c in list(cols):
        c = str(c)
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append("%s__dup%d" % (c, seen[c]))
    return out


def pick_existing(df, cols):
    return [c for c in cols if c in df.columns]


def is_road_class_share_col(c):
    return re.match(r"^road_class_share_.+_lenw_mean$", str(c)) is not None


def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s in {"", "nan", "None", "<NA>", "[]"}:
        return ""
    return s


def choose_road_class_reference(df, road_cols):
    road_cols = [c for c in road_cols if c in df.columns]
    if len(road_cols) <= 1:
        return None
    if ROAD_CLASS_REFERENCE_MODE == "preference_then_largest":
        for c in ROAD_CLASS_REFERENCE_PREFERENCE:
            if c in road_cols:
                return c
    means = df[road_cols].apply(pd.to_numeric, errors="coerce").mean().sort_values(ascending=False)
    if ROAD_CLASS_REFERENCE_MODE == "auto_smallest":
        means = means.sort_values(ascending=True)
    if len(means) == 0:
        return road_cols[-1]
    return str(means.index[0])


def transform_rule_for_variable(var, manual_rules):
    if var in manual_rules:
        return manual_rules.get(var, "raw")
    for pat, rule in PATTERN_VARIABLE_TRANSFORMS.items():
        if re.search(pat, str(var)):
            return rule
    return "raw"


def drop_constant_columns(df):
    keep = []
    dropped = []
    for c in df.columns:
        nun = df[c].nunique(dropna=True)
        if nun <= 1:
            dropped.append(c)
        else:
            keep.append(c)
    return df[keep].copy(), dropped


def numeric_summary_table(df, cols, y_col=None):
    rows = []
    for c in cols:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        non = s.replace([np.inf, -np.inf], np.nan).dropna()
        if len(non) == 0:
            rows.append({
                "variable": c,
                "role": "y" if c == y_col else "x",
                "n": 0,
                "missing": int(len(df)),
                "missing_share": 1.0,
                "n_unique": 0,
                "mean": np.nan,
                "std": np.nan,
                "min": np.nan,
                "p01": np.nan,
                "p05": np.nan,
                "p25": np.nan,
                "p50": np.nan,
                "p75": np.nan,
                "p95": np.nan,
                "p99": np.nan,
                "max": np.nan,
                "share_zero": np.nan,
                "skew": np.nan,
                "kurtosis": np.nan,
                "corr_with_y": np.nan,
            })
            continue
        corr_with_y = np.nan
        if y_col is not None and c != y_col and y_col in df.columns:
            yy = pd.to_numeric(df[y_col], errors="coerce")
            tmp = pd.DataFrame({"x": s, "y": yy}).replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) > 2 and tmp["x"].nunique() > 1 and tmp["y"].nunique() > 1:
                corr_with_y = float(tmp["x"].corr(tmp["y"]))
        rows.append({
            "variable": c,
            "role": "y" if c == y_col else "x",
            "n": int(non.shape[0]),
            "missing": int(s.isna().sum()),
            "missing_share": float(s.isna().mean()),
            "n_unique": int(non.nunique(dropna=True)),
            "mean": float(non.mean()),
            "std": float(non.std(ddof=0)),
            "min": float(non.min()),
            "p01": float(non.quantile(0.01)),
            "p05": float(non.quantile(0.05)),
            "p25": float(non.quantile(0.25)),
            "p50": float(non.quantile(0.50)),
            "p75": float(non.quantile(0.75)),
            "p95": float(non.quantile(0.95)),
            "p99": float(non.quantile(0.99)),
            "max": float(non.max()),
            "share_zero": float((non == 0).mean()),
            "skew": float(non.skew()) if non.nunique() > 2 else np.nan,
            "kurtosis": float(non.kurtosis()) if non.nunique() > 2 else np.nan,
            "corr_with_y": corr_with_y,
        })
    return pd.DataFrame(rows)


In [4]:
# =========================================================
# 3. Transformation and design-matrix helpers
# =========================================================
def _number_label(x):
    v = float(x)
    s = "%g" % v
    s = s.replace("-", "m").replace(".", "p")
    return s


def _transform_one_rule(s, rule):
    rule = str(rule or "raw").strip().lower()

    if rule == "raw":
        return s, "raw"

    if rule.startswith("div:"):
        denom = float(rule.split(":", 1)[1])
        if denom == 0:
            raise ValueError("div transform has zero denominator")
        return s / denom, "div:%g" % denom

    if rule.startswith("mul:"):
        factor = float(rule.split(":", 1)[1])
        return s * factor, "mul:%g" % factor

    if rule.startswith("add:"):
        offset = float(rule.split(":", 1)[1])
        return s + offset, "add:%g" % offset

    if rule == "log":
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = s > 0
        out.loc[mask] = np.log(s.loc[mask])
        return out, "log_positive_only"

    if rule == "log1p":
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = s >= 0
        out.loc[mask] = np.log1p(s.loc[mask])
        return out, "log1p_nonnegative_only"

    if rule == "sqrt":
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = s >= 0
        out.loc[mask] = np.sqrt(s.loc[mask])
        return out, "sqrt_nonnegative_only"

    if rule == "signed_log1p":
        return np.sign(s) * np.log1p(np.abs(s)), "signed_log1p"

    if rule == "logit01":
        eps = 1e-6
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = (s >= 0) & (s <= 1)
        v = s.loc[mask].clip(eps, 1.0 - eps)
        out.loc[mask] = np.log(v / (1.0 - v))
        return out, "logit01_clipped"

    if rule == "standardize":
        mu = s.mean()
        sd = s.std(ddof=0)
        if pd.isna(sd) or sd <= 0:
            return s * np.nan, "standardize_failed"
        return (s - mu) / sd, "standardize"

    raise ValueError("Unknown transform rule: %s" % rule)


def transform_series(s, rule):
    s = pd.to_numeric(s, errors="coerce")
    rules = [r.strip().lower() for r in str(rule or "raw").split("|") if r.strip()]
    if len(rules) == 0:
        rules = ["raw"]

    out = s
    statuses = []
    for one_rule in rules:
        out, status = _transform_one_rule(out, one_rule)
        statuses.append(status)

    return out, "|".join(statuses)


def _rule_prefix(rule):
    rule = str(rule or "raw").strip().lower()

    simple_prefixes = {
        "raw": "",
        "log": "ln_",
        "log1p": "ln1p_",
        "sqrt": "sqrt_",
        "signed_log1p": "slog1p_",
        "logit01": "logit_",
        "standardize": "z_",
    }

    if rule in simple_prefixes:
        return simple_prefixes[rule]

    if rule.startswith("div:"):
        return "div%s_" % _number_label(rule.split(":", 1)[1])

    if rule.startswith("mul:"):
        return "mul%s_" % _number_label(rule.split(":", 1)[1])

    if rule.startswith("add:"):
        return "add%s_" % _number_label(rule.split(":", 1)[1])

    return re.sub(r"[^A-Za-z0-9]+", "_", rule).strip("_") + "_"


def transformed_name(var, rule):
    rule = str(rule or "raw").strip().lower()
    if rule == "raw":
        return var

    rules = [r.strip().lower() for r in rule.split("|") if r.strip()]
    prefix = "".join([_rule_prefix(r) for r in rules])
    return prefix + var


def recommend_transform_for_series(series):
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return "raw", "all_missing"
    nun = int(s.nunique(dropna=True))
    mn = float(s.min())
    if nun <= 2:
        return "raw", "binary_or_constant"
    if nun < RECOMMEND_MIN_UNIQUE:
        return "raw", "low_cardinality"
    skew = float(s.skew()) if nun > 2 else 0.0
    if mn >= 0 and abs(skew) >= RECOMMEND_SKEW_THRESHOLD:
        if mn == 0:
            return "log1p", "nonnegative_with_zero_and_high_skew"
        return "log", "positive_and_high_skew"
    if mn < 0 and abs(skew) >= RECOMMEND_SKEW_THRESHOLD:
        return "signed_log1p", "has_negative_values_and_high_skew"
    return "raw", "no_strong_transform_signal"


def build_transform_audit(df, variables, manual_rules):
    rows = []
    for c in variables:
        if c not in df.columns:
            continue
        rec, reason = recommend_transform_for_series(df[c])
        manual = transform_rule_for_variable(c, manual_rules)
        final = manual
        if AUTO_APPLY_RECOMMENDED_TRANSFORMS and manual == "raw":
            final = rec
        rows.append({
            "variable": c,
            "manual_rule": manual,
            "recommended_rule": rec,
            "recommend_reason": reason,
            "final_rule_if_applied": final,
        })
    return pd.DataFrame(rows)


def apply_transform_rules(df, variables, manual_rules, transform_audit=None):
    out = df.copy()
    rows = []
    final_rules = {}
    audit_map = {}
    if transform_audit is not None and len(transform_audit) > 0:
        for r in transform_audit.itertuples(index=False):
            audit_map[getattr(r, "variable")] = getattr(r, "final_rule_if_applied")
    for c in variables:
        if c not in out.columns:
            continue
        rule = transform_rule_for_variable(c, manual_rules)
        if AUTO_APPLY_RECOMMENDED_TRANSFORMS and c in audit_map and rule == "raw":
            rule = audit_map[c]
        final_rules[c] = rule
        new_c = transformed_name(c, rule)
        if rule == "raw":
            rows.append({
                "variable": c,
                "new_variable": c,
                "rule": "raw",
                "status": "kept_raw",
                "n_missing_after": int(out[c].isna().sum()),
            })
            continue
        try:
            transformed, status = transform_series(out[c], rule)
            out[new_c] = transformed
            rows.append({
                "variable": c,
                "new_variable": new_c,
                "rule": rule,
                "status": status,
                "n_missing_after": int(pd.isna(transformed).sum()),
            })
        except Exception as exc:
            rows.append({
                "variable": c,
                "new_variable": c,
                "rule": rule,
                "status": "failed_%s" % repr(exc),
                "n_missing_after": int(out[c].isna().sum()),
            })
            final_rules[c] = "raw"
    return out, final_rules, pd.DataFrame(rows)


def replace_with_transformed_names(cols, rules):
    out = []
    for c in cols:
        rule = rules.get(c, "raw")
        out.append(transformed_name(c, rule))
    return out


def model_var_for_raw(raw, rules):
    return transformed_name(raw, rules.get(raw, "raw"))


def alias_keep_model_names(rules):
    out = []
    for raw in ALIAS_KEEP_PREFERENCE_RAW:
        out.append(model_var_for_raw(raw, rules))
    return out


def reorder_for_alias_preference(cols, preferred):
    preferred = [c for c in preferred if c in cols]
    rest = [c for c in cols if c not in set(preferred)]
    return preferred + rest


def build_design_matrix(df, cont_cols, cat_cols=None, add_const=True, preferred_order=None):
    cat_cols = cat_cols or []
    preferred_order = preferred_order or []
    cont_cols = reorder_for_alias_preference(list(cont_cols), preferred_order)
    parts = []
    if len(cont_cols) > 0:
        cont_df = df[cont_cols].apply(pd.to_numeric, errors="coerce")
        if cont_df.columns.duplicated().any():
            cont_df.columns = make_unique_column_names(cont_df.columns)
        parts.append(cont_df)
    for c in cat_cols:
        if c not in df.columns:
            continue
        tmp = df[c]
        if pd.api.types.is_numeric_dtype(tmp):
            tmp = tmp.astype("Int64").astype(str)
        else:
            tmp = tmp.astype(str)
        dummies = pd.get_dummies(tmp, prefix=c, drop_first=True)
        if dummies.columns.duplicated().any():
            dummies.columns = make_unique_column_names(dummies.columns)
        parts.append(dummies)
    if len(parts) == 0:
        X = pd.DataFrame(index=df.index)
    else:
        X = pd.concat(parts, axis=1)
    X = X.apply(pd.to_numeric, errors="coerce")
    X, dropped_const = drop_constant_columns(X)
    if add_const:
        X = sm.add_constant(X, has_constant="add")
    return X, dropped_const


def drop_exact_alias_columns(X, protect_cols=None, tol=1e-12):
    protect_cols = protect_cols or []
    ordered_cols = [c for c in X.columns if c in protect_cols] + [c for c in X.columns if c not in protect_cols]
    X = X[ordered_cols].copy()
    if X.shape[1] <= 1:
        return X, []
    vals = X.to_numpy(dtype=float)
    keep_idx = []
    dropped = []
    current_rank = 0
    for j, c in enumerate(X.columns):
        sub_idx = keep_idx + [j]
        sub_vals = vals[:, sub_idx]
        try:
            rank = int(np.linalg.matrix_rank(sub_vals, tol=tol))
        except Exception:
            rank = current_rank
        if rank > current_rank:
            keep_idx.append(j)
            current_rank = rank
        else:
            dropped.append(c)
    keep_cols = list(X.columns[keep_idx])
    return X[keep_cols].copy(), dropped


def prepare_vif_matrix(X, max_rows=None):
    """
    Prepare a numeric X matrix for VIF diagnostics.

    Important: this function does not remove the intercept before the
    auxiliary regressions. The intercept is included in each auxiliary
    regression to avoid uncentered VIF inflation for positive variables.
    """
    X_vif = X.copy()
    X_vif = X_vif.apply(pd.to_numeric, errors="coerce")
    X_vif = X_vif.replace([np.inf, -np.inf], np.nan)

    # VIF should use complete cases. This matches the actual OLS design
    # after rows with missing X have been removed.
    X_vif = X_vif.dropna(axis=0, how="any")

    if max_rows is not None and len(X_vif) > max_rows:
        X_vif = X_vif.sample(max_rows, random_state=RANDOM_SEED)

    # Drop constant non-intercept columns. Keep const if present.
    keep_cols = []
    dropped_constant = []
    for c in X_vif.columns:
        if c == "const":
            keep_cols.append(c)
            continue
        nun = X_vif[c].nunique(dropna=True)
        if nun <= 1:
            dropped_constant.append(c)
        else:
            keep_cols.append(c)

    X_vif = X_vif[keep_cols].copy()
    return X_vif, dropped_constant


def compute_vif(X, max_rows=None):
    """
    Correct VIF calculation with an intercept in each auxiliary regression.

    This implementation uses numpy least squares instead of statsmodels inside
    the loop, which is much faster once interactions create a wider model matrix.
    """
    X_vif, dropped_constant = prepare_vif_matrix(X, max_rows=max_rows)

    if "const" not in X_vif.columns:
        X_vif = sm.add_constant(X_vif, has_constant="add")

    variable_cols = [c for c in X_vif.columns if c != "const"]
    if len(variable_cols) == 0:
        return pd.DataFrame(columns=[
            "variable", "VIF", "aux_r2", "n_obs", "n_predictors_in_aux", "status"
        ])

    X_vif = X_vif.astype(float)
    rows = []
    for target in variable_cols:
        y_aux = X_vif[target].to_numpy(dtype=float)
        other_cols = [c for c in X_vif.columns if c != target]
        Z_aux = X_vif[other_cols].to_numpy(dtype=float)

        if Z_aux.shape[1] == 0:
            rows.append({
                "variable": target,
                "VIF": 1.0,
                "aux_r2": 0.0,
                "n_obs": int(len(X_vif)),
                "n_predictors_in_aux": 0,
                "status": "only_variable",
            })
            continue

        try:
            beta, residuals, rank, svals = np.linalg.lstsq(Z_aux, y_aux, rcond=None)
            fitted = Z_aux.dot(beta)
            resid = y_aux - fitted
            centered = y_aux - np.nanmean(y_aux)
            ss_total = float(np.dot(centered, centered))
            ss_resid = float(np.dot(resid, resid))
            if ss_total <= 0 or not np.isfinite(ss_total):
                aux_r2 = np.nan
                vif = np.nan
                status = "constant_target"
            else:
                aux_r2 = max(0.0, min(1.0, 1.0 - ss_resid / ss_total))
                if aux_r2 >= 1.0 - 1e-12:
                    vif = np.inf
                else:
                    vif = 1.0 / (1.0 - aux_r2)
                status = "ok" if rank == Z_aux.shape[1] else "ok_rank_deficient_aux"
        except Exception as exc:
            aux_r2 = np.nan
            vif = np.nan
            status = "failed: %s" % str(exc)

        rows.append({
            "variable": target,
            "VIF": float(vif) if pd.notna(vif) else np.nan,
            "aux_r2": aux_r2,
            "n_obs": int(len(X_vif)),
            "n_predictors_in_aux": int(max(0, Z_aux.shape[1] - 1)),
            "status": status,
        })

    vif_df = pd.DataFrame(rows)
    if len(dropped_constant) > 0:
        dropped_df = pd.DataFrame({
            "variable": dropped_constant,
            "VIF": np.nan,
            "aux_r2": np.nan,
            "n_obs": int(len(X_vif)),
            "n_predictors_in_aux": np.nan,
            "status": "dropped_constant_before_vif",
        })
        vif_df = pd.concat([vif_df, dropped_df], ignore_index=True)

    return vif_df.sort_values(["VIF", "variable"], ascending=[False, True]).reset_index(drop=True)


def iterative_drop_high_vif(X, threshold=10.0, protected=None):
    protected = set(protected or [])
    X_cur = X.copy()
    dropped = []

    while True:
        vif = compute_vif(X_cur)
        if vif.empty:
            break

        vif_ok = vif[vif["status"] == "ok"].copy()
        vif_ok = vif_ok[~vif_ok["variable"].isin(protected)].copy()
        if vif_ok.empty:
            break

        top = vif_ok.iloc[0]
        top_vif = float(top["VIF"])
        if np.isfinite(top_vif) and top_vif > threshold:
            col = top["variable"]
            if col in X_cur.columns:
                X_cur = X_cur.drop(columns=[col])
                dropped.append({
                    "variable": col,
                    "VIF_at_drop": top_vif,
                    "aux_r2_at_drop": float(top.get("aux_r2", np.nan)),
                })
                continue

        if not np.isfinite(top_vif):
            col = top["variable"]
            if col in X_cur.columns:
                X_cur = X_cur.drop(columns=[col])
                dropped.append({
                    "variable": col,
                    "VIF_at_drop": top_vif,
                    "aux_r2_at_drop": float(top.get("aux_r2", np.nan)),
                })
                continue

        break

    return X_cur, pd.DataFrame(dropped)


def vif_pairwise_correlations(X, threshold=0.60):
    X_vif, _ = prepare_vif_matrix(X)
    cols = [c for c in X_vif.columns if c != "const"]
    if len(cols) == 0:
        corr = pd.DataFrame()
        pairs = pd.DataFrame(columns=["var1", "var2", "corr", "abs_corr"])
        return corr, pairs

    corr = X_vif[cols].corr()
    rows = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            val = corr.loc[a, b]
            if pd.notna(val) and abs(val) >= threshold:
                rows.append({
                    "var1": a,
                    "var2": b,
                    "corr": float(val),
                    "abs_corr": float(abs(val)),
                })
    pairs = pd.DataFrame(rows)
    if len(pairs) > 0:
        pairs = pairs.sort_values("abs_corr", ascending=False).reset_index(drop=True)
    else:
        pairs = pd.DataFrame(columns=["var1", "var2", "corr", "abs_corr"])
    return corr, pairs


def standardize_series_for_vif(s):
    s = pd.to_numeric(s, errors="coerce")
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s * 0
    return (s - s.mean()) / sd


def _fast_aux_r2(y, Z):
    y_arr = np.asarray(y, dtype=float)
    Z_arr = np.asarray(Z, dtype=float)
    if Z_arr.ndim == 1:
        Z_arr = Z_arr.reshape(-1, 1)
    if Z_arr.shape[1] == 0:
        return np.nan
    try:
        beta, residuals, rank, svals = np.linalg.lstsq(Z_arr, y_arr, rcond=None)
        fitted = Z_arr.dot(beta)
        resid = y_arr - fitted
        centered = y_arr - np.nanmean(y_arr)
        ss_total = float(np.dot(centered, centered))
        ss_resid = float(np.dot(resid, resid))
        if ss_total <= 0 or not np.isfinite(ss_total):
            return np.nan
        return max(0.0, min(1.0, 1.0 - ss_resid / ss_total))
    except Exception:
        return np.nan


def auxiliary_regression_drivers(X, target_var, top_n=10):
    X_vif, _ = prepare_vif_matrix(X)
    if "const" in X_vif.columns:
        X_vif = X_vif.drop(columns=["const"])

    if target_var not in X_vif.columns:
        return pd.DataFrame(), np.nan, 0

    y = X_vif[target_var].astype(float)
    other_cols = [c for c in X_vif.columns if c != target_var]
    if len(other_cols) == 0:
        return pd.DataFrame(), np.nan, int(len(X_vif))

    Z = X_vif[other_cols].astype(float)
    keep_cols = [c for c in Z.columns if Z[c].nunique(dropna=True) > 1]
    Z = Z[keep_cols]
    if Z.shape[1] == 0:
        return pd.DataFrame(), np.nan, int(len(X_vif))

    Z_const = sm.add_constant(Z, has_constant="add")
    aux_r2_full = _fast_aux_r2(y.to_numpy(dtype=float), Z_const.to_numpy(dtype=float))
    if pd.isna(aux_r2_full):
        return pd.DataFrame(), np.nan, int(len(X_vif))

    y_std = standardize_series_for_vif(y)
    Z_std = Z.apply(standardize_series_for_vif)
    Z_std_const = sm.add_constant(Z_std, has_constant="add")
    beta_std_map = {}
    try:
        beta_std, residuals, rank, svals = np.linalg.lstsq(
            Z_std_const.to_numpy(dtype=float),
            y_std.to_numpy(dtype=float),
            rcond=None,
        )
        beta_std_map = dict(zip(Z_std_const.columns, beta_std))
    except Exception:
        beta_std_map = {}

    rows = []
    for c in Z.columns:
        corr_val = np.nan
        try:
            corr_val = float(y.corr(Z[c]))
        except Exception:
            pass

        beta_std = beta_std_map.get(c, np.nan)

        rows.append({
            "target_vif_variable": target_var,
            "driver_variable": c,
            "pairwise_corr_with_target": corr_val,
            "abs_pairwise_corr": abs(corr_val) if pd.notna(corr_val) else np.nan,
            "aux_standardized_beta": beta_std,
            "abs_aux_standardized_beta": abs(beta_std) if pd.notna(beta_std) else np.nan,
            "aux_r2_full": aux_r2_full,
            "n_obs": int(len(X_vif)),
        })

    drivers = pd.DataFrame(rows)
    if len(drivers) == 0:
        return drivers, aux_r2_full, int(len(X_vif))

    top_candidates = (
        drivers
        .sort_values(["abs_pairwise_corr", "abs_aux_standardized_beta"], ascending=False)
        .head(top_n)["driver_variable"]
        .tolist()
    )

    delta_map = {}
    for c in top_candidates:
        reduced_cols = [x for x in Z.columns if x != c]
        if len(reduced_cols) == 0:
            delta_map[c] = aux_r2_full
            continue
        try:
            Z_reduced_const = sm.add_constant(Z[reduced_cols], has_constant="add")
            aux_r2_red = _fast_aux_r2(y.to_numpy(dtype=float), Z_reduced_const.to_numpy(dtype=float))
            delta_map[c] = float(aux_r2_full - aux_r2_red) if pd.notna(aux_r2_red) else np.nan
        except Exception:
            delta_map[c] = np.nan

    drivers["aux_delta_r2_if_removed"] = drivers["driver_variable"].map(delta_map).fillna(0.0)
    drivers = drivers.sort_values(
        ["aux_delta_r2_if_removed", "abs_pairwise_corr", "abs_aux_standardized_beta"],
        ascending=False,
    ).reset_index(drop=True)
    return drivers, aux_r2_full, int(len(X_vif))


def save_vif_driver_diagnostics(X, vif_df, relative_dir="diagnostics/vif_drivers", corr_threshold=0.60, top_n=10):
    """Save pairwise-correlation and auxiliary-regression diagnostics for VIF."""
    out_rel = Path(relative_dir)
    plot_dir = PLOTS_DIR / "vif_drivers"
    ensure_dir(plot_dir)

    corr, high_pairs = vif_pairwise_correlations(X, threshold=corr_threshold)
    if len(corr) > 0:
        write_result_table(corr.reset_index().rename(columns={"index": "variable"}), out_rel / "vif_pairwise_correlation_matrix.csv", index=False, rounded=True)
    else:
        write_result_table(pd.DataFrame(), out_rel / "vif_pairwise_correlation_matrix.csv", index=False, rounded=True)
    write_result_table(high_pairs, out_rel / "vif_high_pairwise_correlations.csv", index=False, rounded=True)

    if len(corr) > 0:
        labels = list(corr.columns)
        n = len(labels)
        fig_w = max(10, min(28, 0.55 * n))
        fig_h = max(8, min(28, 0.55 * n))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(corr.values, vmin=-1, vmax=1)
        ax.set_xticks(np.arange(n))
        ax.set_yticks(np.arange(n))
        ax.set_xticklabels(labels, rotation=90, fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Correlation")
        ax.set_title("Pairwise Correlation among Model X Variables")
        plt.tight_layout()
        fig.savefig(plot_dir / "vif_pairwise_correlation_heatmap.png", dpi=220, bbox_inches="tight")
        plt.close(fig)

    if vif_df is None or len(vif_df) == 0:
        driver_summary = pd.DataFrame()
        driver_long = pd.DataFrame()
        write_result_table(driver_summary, out_rel / "vif_driver_summary.csv", index=False, rounded=True)
        write_result_table(driver_long, out_rel / "vif_driver_long.csv", index=False, rounded=True)
        return driver_summary, driver_long, high_pairs

    vif_ok = vif_df[vif_df["status"] == "ok"].copy() if "status" in vif_df.columns else vif_df.copy()
    if "VIF" not in vif_ok.columns:
        driver_summary = pd.DataFrame()
        driver_long = pd.DataFrame()
        write_result_table(driver_summary, out_rel / "vif_driver_summary.csv", index=False, rounded=True)
        write_result_table(driver_long, out_rel / "vif_driver_long.csv", index=False, rounded=True)
        return driver_summary, driver_long, high_pairs

    high_vif_vars = vif_ok[vif_ok["VIF"] >= 5.0].sort_values("VIF", ascending=False)["variable"].tolist()
    top_vif_vars = vif_ok.sort_values("VIF", ascending=False).head(5)["variable"].tolist()
    diag_targets = list(OrderedDict.fromkeys(high_vif_vars + top_vif_vars))

    driver_tables = []
    summary_rows = []
    for target in diag_targets:
        drivers, aux_r2, n_obs = auxiliary_regression_drivers(X, target, top_n=top_n)
        target_vif_s = vif_ok.loc[vif_ok["variable"] == target, "VIF"]
        target_vif = float(target_vif_s.iloc[0]) if len(target_vif_s) else np.nan
        summary_rows.append({
            "target_vif_variable": target,
            "VIF": target_vif,
            "aux_r2": aux_r2,
            "n_obs": n_obs,
            "top_driver_by_delta_r2": drivers["driver_variable"].iloc[0] if len(drivers) else "",
            "top_delta_r2": drivers["aux_delta_r2_if_removed"].iloc[0] if len(drivers) else np.nan,
            "top_pairwise_corr_driver": drivers.sort_values("abs_pairwise_corr", ascending=False)["driver_variable"].iloc[0] if len(drivers) else "",
            "top_abs_pairwise_corr": drivers.sort_values("abs_pairwise_corr", ascending=False)["abs_pairwise_corr"].iloc[0] if len(drivers) else np.nan,
        })
        if len(drivers) > 0:
            driver_tables.append(drivers)

    driver_summary = pd.DataFrame(summary_rows)
    if len(driver_summary) > 0:
        driver_summary = driver_summary.sort_values("VIF", ascending=False).reset_index(drop=True)
    driver_long = pd.concat(driver_tables, ignore_index=True) if len(driver_tables) else pd.DataFrame()

    write_result_table(driver_summary, out_rel / "vif_driver_summary.csv", index=False, rounded=True)
    write_result_table(driver_long, out_rel / "vif_driver_long.csv", index=False, rounded=True)
    return driver_summary, driver_long, high_pairs


def standardize_columns(X, cols):
    X = X.copy()
    rows = []
    for c in cols:
        if c not in X.columns or c == "const":
            continue
        s = pd.to_numeric(X[c], errors="coerce")
        mu = float(s.mean())
        sd = float(s.std(ddof=0))
        if not np.isfinite(sd) or sd <= 0:
            continue
        X[c] = (s - mu) / sd
        rows.append({"column": c, "mean": mu, "std": sd})
    return X, pd.DataFrame(rows)


# ---------------------------------------------------------
# Derived variables and interaction helpers
# ---------------------------------------------------------
def add_derived_variables(df, specs):
    out = df.copy()
    rows = []

    for new_col, spec in specs.items():
        kind = spec.get("type")

        if kind == "cut":
            src = spec.get("source")
            if src not in out.columns:
                rows.append({
                    "new_variable": new_col,
                    "type": kind,
                    "source": src,
                    "status": "missing_source",
                })
                continue

            out[new_col] = pd.cut(
                pd.to_numeric(out[src], errors="coerce"),
                bins=spec["bins"],
                labels=spec["labels"],
                include_lowest=True,
            ).astype(str)
            out.loc[out[new_col].isin(["nan", "NaN", "<NA>"]), new_col] = np.nan

            rows.append({
                "new_variable": new_col,
                "type": kind,
                "source": src,
                "status": "created",
            })

        elif kind == "indicator":
            condition = spec.get("condition")
            try:
                mask = out.eval(condition)
                out[new_col] = mask.astype(int)
                rows.append({
                    "new_variable": new_col,
                    "type": kind,
                    "source": condition,
                    "status": "created",
                })
            except Exception as exc:
                out[new_col] = np.nan
                rows.append({
                    "new_variable": new_col,
                    "type": kind,
                    "source": condition,
                    "status": "failed_%s" % repr(exc),
                })

        else:
            rows.append({
                "new_variable": new_col,
                "type": kind,
                "source": spec.get("source", spec.get("condition", "")),
                "status": "unknown_type",
            })

    return out, pd.DataFrame(rows)


def safe_interaction_name(a, b):
    name = "%s_x_%s" % (str(a), str(b))
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name[:220]


def categorical_dummy_frame_for_interaction(df, col):
    tmp = df[col]
    if pd.api.types.is_numeric_dtype(tmp):
        tmp = tmp.astype("Int64").astype(str)
    else:
        tmp = tmp.astype(str)
    dummies = pd.get_dummies(tmp, prefix=col, drop_first=True)
    dummies = dummies.apply(pd.to_numeric, errors="coerce")
    if dummies.columns.duplicated().any():
        dummies.columns = make_unique_column_names(dummies.columns)
    return dummies


def expand_interaction_terms(df, x_model, cat_cols, x_rules, specs):
    out = df.copy()
    x_model_out = list(x_model)
    rows = []

    if not ADD_INTERACTIONS:
        return out, x_model_out, pd.DataFrame(rows)

    cat_dummy_cache = {}

    for spec in specs:
        spec_name = spec.get("name", "interaction")
        left_vars = spec.get("left", [])
        right_vars = spec.get("right", [])

        for left in left_vars:
            if left not in out.columns:
                rows.append({
                    "interaction_group": spec_name,
                    "left": left,
                    "right": "",
                    "new_variable": "",
                    "status": "missing_left",
                })
                continue

            if left in cat_cols:
                if left not in cat_dummy_cache:
                    cat_dummy_cache[left] = categorical_dummy_frame_for_interaction(out, left)
                left_df = cat_dummy_cache[left]
                for c in left_df.columns:
                    if c not in out.columns:
                        out[c] = left_df[c]
                left_model_vars = list(left_df.columns)
            else:
                left_model = transformed_name(left, x_rules.get(left, "raw"))
                if left_model not in out.columns:
                    rows.append({
                        "interaction_group": spec_name,
                        "left": left,
                        "right": "",
                        "new_variable": "",
                        "status": "missing_left_model",
                    })
                    continue
                left_model_vars = [left_model]

            for right in right_vars:
                if right not in out.columns and transformed_name(right, x_rules.get(right, "raw")) not in out.columns:
                    rows.append({
                        "interaction_group": spec_name,
                        "left": left,
                        "right": right,
                        "new_variable": "",
                        "status": "missing_right",
                    })
                    continue

                right_model = transformed_name(right, x_rules.get(right, "raw"))
                if right_model not in out.columns:
                    rows.append({
                        "interaction_group": spec_name,
                        "left": left,
                        "right": right,
                        "new_variable": "",
                        "status": "missing_right_model",
                    })
                    continue

                right_s = pd.to_numeric(out[right_model], errors="coerce")

                for left_model in left_model_vars:
                    if left_model not in out.columns:
                        rows.append({
                            "interaction_group": spec_name,
                            "left": left,
                            "right": right,
                            "new_variable": "",
                            "status": "missing_materialized_left",
                        })
                        continue

                    base_new_col = safe_interaction_name(left_model, right_model)
                    new_col = base_new_col
                    k = 2
                    while new_col in out.columns and new_col not in x_model_out:
                        new_col = "%s_%d" % (base_new_col, k)
                        k += 1

                    out[new_col] = pd.to_numeric(out[left_model], errors="coerce") * right_s

                    if new_col not in x_model_out:
                        x_model_out.append(new_col)

                    rows.append({
                        "interaction_group": spec_name,
                        "left": left,
                        "left_model": left_model,
                        "right": right,
                        "right_model": right_model,
                        "new_variable": new_col,
                        "status": "created",
                    })

    return out, x_model_out, pd.DataFrame(rows)


In [5]:
# =========================================================
# 4. Load table and prepare modeling data
# =========================================================

def model_add_action_context_columns(df):
    """Backfill action-context columns if the variable table was built before they existed."""
    out = df.copy()

    def norm_action_series(s):
        z = s.astype(str).str.strip().str.upper()
        z = z.replace({"": np.nan, "NAN": np.nan, "NONE": np.nan, "<NA>": np.nan})
        return z

    for col in ["start_action", "end_action"]:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = norm_action_series(out[col])

    if "parent_start_action" not in out.columns:
        out["parent_start_action"] = out["start_action"]
    else:
        out["parent_start_action"] = norm_action_series(out["parent_start_action"]).fillna(out["start_action"])

    if "parent_end_action" not in out.columns:
        out["parent_end_action"] = out["end_action"]
    else:
        out["parent_end_action"] = norm_action_series(out["parent_end_action"]).fillna(out["end_action"])

    out["action_pair"] = out["start_action"].astype(str) + " -> " + out["end_action"].astype(str)
    out["parent_action_pair"] = out["parent_start_action"].astype(str) + " -> " + out["parent_end_action"].astype(str)

    out["segment_target_action_inferred"] = out["end_action"]
    end_is_grab = out["end_action"].eq("GRAB")
    parent_target_is_task = out["parent_end_action"].isin(["FETCH", "DELIVER"])
    out.loc[end_is_grab & parent_target_is_task, "segment_target_action_inferred"] = out.loc[
        end_is_grab & parent_target_is_task,
        "parent_end_action",
    ]

    out["segment_task_orientation"] = "unknown_or_other"
    out.loc[out["segment_target_action_inferred"].eq("FETCH"), "segment_task_orientation"] = "to_fetch"
    out.loc[out["segment_target_action_inferred"].eq("DELIVER"), "segment_task_orientation"] = "to_deliver"

    out["is_grab_interruption"] = out["end_action"].eq("GRAB").astype(int)
    out["is_after_grab_update"] = out["start_action"].eq("GRAB").astype(int)
    out["has_grab_endpoint"] = (
        out["start_action"].eq("GRAB") | out["end_action"].eq("GRAB")
    ).astype(int)

    out["segment_grab_context"] = "no_grab_endpoint"
    out.loc[out["end_action"].eq("GRAB"), "segment_grab_context"] = "pre_grab_interrupted"
    out.loc[out["start_action"].eq("GRAB"), "segment_grab_context"] = "post_grab_after_update"
    out.loc[
        out["start_action"].eq("GRAB") & out["end_action"].eq("GRAB"),
        "segment_grab_context",
    ] = "between_grab_events"

    return out


def protected_non_x():
    out = list(PROTECTED_NON_X_BASE)
    out.append(Y_COL)
    if DROP_MECHANICAL_X_WHEN_Y_IS_SPEED and Y_COL == "speed_kmh":
        out.extend(MECHANICAL_X_WHEN_Y_SPEED)
    return list(OrderedDict.fromkeys(out))


def load_and_filter_trip_table(row_query=None):
    df = read_table(TRIP_TABLE_PATH)
    if "action_pair" not in df.columns and {"start_action", "end_action"}.issubset(df.columns):
        df["action_pair"] = df["start_action"].astype(str) + " -> " + df["end_action"].astype(str)
    df = model_add_action_context_columns(df)
    print("Raw modeling table shape:", df.shape)
    print("Action context columns:")
    print(df[["segment_task_orientation", "segment_grab_context"]].value_counts(dropna=False).to_string())
    if row_query is not None:
        before = len(df)
        df = df.query(row_query).copy()
        print("After row query: %s -> %s" % (before, len(df)))
    if Y_COL not in df.columns:
        raise KeyError("Y_COL not found in input table: %s" % Y_COL)
    return df


def flatten_whitelist_groups(groups, existing_cols):
    existing_cols = set(existing_cols)
    rows = []
    selected = []
    missing = []
    for group_name, cols in groups.items():
        for c in cols:
            if c in existing_cols:
                if c not in selected:
                    selected.append(c)
                rows.append({"group_name": group_name, "variable": c, "exists": 1})
            else:
                missing.append({"group_name": group_name, "variable": c, "exists": 0})
                rows.append({"group_name": group_name, "variable": c, "exists": 0})
    return selected, pd.DataFrame(rows), pd.DataFrame(missing)


def prepare_modeling_data(df):
    df, derived_info = add_derived_variables(df, DERIVED_VARIABLE_SPECS)

    x_raw, selected_groups_all, missing_whitelist = flatten_whitelist_groups(X_GROUPS, df.columns)

    for c in X_EXTRA:
        if c in df.columns and c not in x_raw:
            x_raw.append(c)
            selected_groups_all = pd.concat([
                selected_groups_all,
                pd.DataFrame([{"group_name": "x_extra", "variable": c, "exists": 1}])
            ], ignore_index=True)
        elif c not in df.columns:
            selected_groups_all = pd.concat([
                selected_groups_all,
                pd.DataFrame([{"group_name": "x_extra", "variable": c, "exists": 0}])
            ], ignore_index=True)
            missing_whitelist = pd.concat([
                missing_whitelist,
                pd.DataFrame([{"group_name": "x_extra", "variable": c, "exists": 0}])
            ], ignore_index=True)

    protected = set(protected_non_x())
    x_drop_set = set(X_DROP) | protected
    x_raw = [c for c in x_raw if c not in x_drop_set]

    road_class_cols_selected = [c for c in x_raw if is_road_class_share_col(c)]
    road_class_reference = None
    if DROP_ONE_ROAD_CLASS_REFERENCE and len(road_class_cols_selected) > 1:
        road_class_reference = choose_road_class_reference(df, road_class_cols_selected)
        if road_class_reference in x_raw:
            x_raw = [c for c in x_raw if c != road_class_reference]
            print("Dropped road-class reference column:", road_class_reference)

    cat_cols = pick_existing(df, CATEGORICAL_VARS)

    selected_groups = selected_groups_all[selected_groups_all["exists"] == 1].copy()
    selected_groups["included_in_x_raw"] = selected_groups["variable"].isin(x_raw).astype(int)

    road_class_diagnostics = pd.DataFrame({
        "road_class_column": road_class_cols_selected,
        "included_in_x_after_reference_drop": [c in x_raw for c in road_class_cols_selected],
        "is_reference": [c == road_class_reference for c in road_class_cols_selected],
        "mean_share": [pd.to_numeric(df[c], errors="coerce").mean() if c in df.columns else np.nan for c in road_class_cols_selected],
    })

    audit_vars = list(OrderedDict.fromkeys(x_raw + [Y_COL]))
    summary = numeric_summary_table(df, audit_vars, y_col=Y_COL)
    transform_audit = build_transform_audit(df, x_raw, VARIABLE_TRANSFORMS)

    df2, x_rules, x_transform_info = apply_transform_rules(
        df,
        variables=x_raw,
        manual_rules=VARIABLE_TRANSFORMS,
        transform_audit=transform_audit,
    )
    x_model = replace_with_transformed_names(x_raw, x_rules)
    x_model = [c for c in x_model if c in df2.columns]

    df2, x_model, interaction_info = expand_interaction_terms(
        df=df2,
        x_model=x_model,
        cat_cols=cat_cols,
        x_rules=x_rules,
        specs=INTERACTION_SPECS,
    )

    y_new = transformed_name(Y_COL, Y_TRANSFORM)
    if Y_TRANSFORM == "raw":
        df2[y_new] = pd.to_numeric(df2[Y_COL], errors="coerce")
        y_info = pd.DataFrame([{
            "variable": Y_COL,
            "new_variable": y_new,
            "rule": "raw",
            "status": "kept_raw",
            "n_missing_after": int(df2[y_new].isna().sum()),
        }])
    else:
        y_transformed, y_status = transform_series(df2[Y_COL], Y_TRANSFORM)
        df2[y_new] = y_transformed
        y_info = pd.DataFrame([{
            "variable": Y_COL,
            "new_variable": y_new,
            "rule": Y_TRANSFORM,
            "status": y_status,
            "n_missing_after": int(df2[y_new].isna().sum()),
        }])

    needed = [y_new] + x_model + cat_cols
    if CLUSTER_COL in df2.columns:
        needed.append(CLUSTER_COL)
    before = len(df2)
    df_model = df2.dropna(subset=[c for c in needed if c in df2.columns]).copy()
    print("Rows after dropping model missing values: %s -> %s" % (before, len(df_model)))
    print("Selected raw X count:", len(x_raw))
    print("Selected model X count:", len(x_model))

    if len(missing_whitelist) > 0:
        print("Missing whitelist variables:", len(missing_whitelist))
        display(missing_whitelist)

    return {
        "df_model": df_model,
        "df_all_transformed": df2,
        "x_raw": x_raw,
        "x_model": x_model,
        "x_rules": x_rules,
        "cat_cols": cat_cols,
        "y_model": y_new,
        "summary": summary,
        "transform_audit": transform_audit,
        "x_transform_info": x_transform_info,
        "y_transform_info": y_info,
        "selected_groups": selected_groups,
        "missing_whitelist": missing_whitelist,
        "road_class_diagnostics": road_class_diagnostics,
        "road_class_reference": road_class_reference,
        "derived_info": derived_info,
        "interaction_info": interaction_info,
    }


def save_preparation_outputs(prep):
    write_result_table(prep["summary"], "diagnostics/selected_variable_raw_summary.csv", index=False, rounded=True)
    write_result_table(prep["transform_audit"], "diagnostics/transform_recommendations.csv", index=False, rounded=False)
    write_result_table(prep["x_transform_info"], "diagnostics/x_transform_info.csv", index=False, rounded=False)
    write_result_table(prep["y_transform_info"], "diagnostics/y_transform_info.csv", index=False, rounded=False)
    write_result_table(prep["selected_groups"], "diagnostics/selected_variables_by_group.csv", index=False, rounded=False)
    write_result_table(prep["missing_whitelist"], "diagnostics/missing_whitelist_variables.csv", index=False, rounded=False)
    write_result_table(prep["road_class_diagnostics"], "diagnostics/road_class_model_diagnostics.csv", index=False, rounded=True)
    write_result_table(prep["derived_info"], "diagnostics/derived_variable_info.csv", index=False, rounded=False)
    write_result_table(prep["interaction_info"], "diagnostics/interaction_info.csv", index=False, rounded=False)
    print("Road-class reference column:", prep["road_class_reference"])
    display(prep["transform_audit"].head(30))


## Distribution dashboard before modeling

This block gives a complete visual check of the selected target and predictors after transformations. It also saves a numeric summary table, correlation with the target, and highly correlated X pairs.


In [6]:
# =========================================================
# 5. Variable distribution dashboard
# =========================================================
def make_variable_distribution_dashboard(prep):
    outdir = DIAGNOSTICS_DIR / "variable_distributions"
    ensure_dir(outdir)

    df_dist = prep["df_model"].copy()
    y_col = prep["y_model"]
    x_cols = list(prep["x_model"])
    plot_cols = [y_col] + x_cols
    plot_cols = [c for c in plot_cols if c in df_dist.columns]

    if len(plot_cols) > MAX_DISTRIBUTION_PLOT_VARS:
        print("Too many variables to plot. Plotting first %d variables." % MAX_DISTRIBUTION_PLOT_VARS)
        plot_cols = plot_cols[:MAX_DISTRIBUTION_PLOT_VARS]

    if len(df_dist) > DISTRIBUTION_PLOT_SAMPLE_N:
        df_plot = df_dist.sample(DISTRIBUTION_PLOT_SAMPLE_N, random_state=RANDOM_SEED).copy()
    else:
        df_plot = df_dist.copy()

    variable_summary = numeric_summary_table(df_dist, plot_cols, y_col=y_col)
    write_result_table(variable_summary, "diagnostics/variable_distributions/model_variable_summary.csv", index=False, rounded=True)
    print("Variable summary saved under:", DIAGNOSTICS_DIR / "variable_distributions")
    display(variable_summary)

    n_vars = len(plot_cols)
    n_cols = 4
    n_rows = int(math.ceil(float(n_vars) / n_cols)) if n_vars else 1
    fig_w = 5.0 * n_cols
    fig_h = max(3.6 * n_rows, 4.0)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(-1)

    for i, c in enumerate(plot_cols):
        ax = axes[i]
        s = pd.to_numeric(df_plot[c], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(s) == 0:
            ax.text(0.5, 0.5, "all missing", ha="center", va="center")
            ax.set_title(c)
            continue
        lo = s.quantile(0.01)
        hi = s.quantile(0.99)
        s_plot = s[(s >= lo) & (s <= hi)]
        if s_plot.nunique() <= 20:
            counts = s_plot.value_counts().sort_index()
            ax.bar(range(len(counts)), counts.values)
            ax.set_xticks(range(len(counts)))
            ax.set_xticklabels([str(x) for x in counts.index], rotation=45, ha="right", fontsize=8)
        else:
            ax.hist(s_plot.values, bins=DISTRIBUTION_HIST_BINS, alpha=0.85)
        title_prefix = "Y: " if c == y_col else "X: "
        ax.set_title(title_prefix + c, fontsize=10)
        ax.tick_params(axis="both", labelsize=8)
        row = variable_summary[variable_summary["variable"] == c]
        if len(row) > 0:
            row = row.iloc[0]
            txt = "p50=%.3g\nzero=%.2f\nskew=%.2f" % (
                row["p50"],
                row["share_zero"] if pd.notna(row["share_zero"]) else np.nan,
                row["skew"] if pd.notna(row["skew"]) else np.nan,
            )
            ax.text(
                0.98,
                0.95,
                txt,
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=8,
                bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
            )

    for j in range(n_vars, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    fig_path = PLOTS_DIR / "model_variable_distribution_dashboard.png"
    plt.savefig(fig_path, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved distribution dashboard:", fig_path)

    numeric_for_corr = df_dist[plot_cols].apply(pd.to_numeric, errors="coerce")
    corr = numeric_for_corr.corr()
    write_result_table(corr.reset_index().rename(columns={"index": "variable"}), "diagnostics/variable_distributions/model_variable_correlation_matrix.csv", index=False, rounded=True)

    if y_col in corr.columns:
        y_corr = (
            corr[y_col]
            .drop(labels=[y_col], errors="ignore")
            .dropna()
            .sort_values(key=lambda s: s.abs(), ascending=False)
            .reset_index()
        )
        y_corr.columns = ["variable", "corr_with_y"]
        write_result_table(y_corr, "diagnostics/variable_distributions/correlation_with_y.csv", index=False, rounded=True)
        print("Top absolute correlations with Y:")
        display(y_corr.head(30))

    pairs = []
    x_numeric_cols = [c for c in x_cols if c in corr.index]
    for i, a in enumerate(x_numeric_cols):
        for b in x_numeric_cols[i + 1:]:
            val = corr.loc[a, b]
            if pd.notna(val) and abs(val) >= 0.85:
                pairs.append({
                    "var1": a,
                    "var2": b,
                    "corr": float(val),
                    "abs_corr": float(abs(val)),
                })
    if len(pairs):
        high_corr_pairs = pd.DataFrame(pairs).sort_values("abs_corr", ascending=False)
    else:
        high_corr_pairs = pd.DataFrame(columns=["var1", "var2", "corr", "abs_corr"])
    write_result_table(high_corr_pairs, "diagnostics/variable_distributions/high_pairwise_corr_x_ge_085.csv", index=False, rounded=True)
    print("High pairwise X correlations, abs(corr) >= 0.85:")
    display(high_corr_pairs.head(50))
    return variable_summary

# Execution is handled in the final orchestration cell.


In [7]:
# =========================================================
# 6. OLS functions and execution
# =========================================================
def matrix_condition_number(X):
    try:
        return float(np.linalg.cond(X.to_numpy(dtype=float)))
    except Exception:
        return np.nan


def coef_table_from_ols(res):
    params = pd.Series(res.params)
    se = pd.Series(res.bse, index=params.index)
    tvals = pd.Series(res.tvalues, index=params.index)
    pvals = pd.Series(res.pvalues, index=params.index)
    ci = res.conf_int()
    if isinstance(ci, np.ndarray):
        ci = pd.DataFrame(ci, index=params.index, columns=["ci_low", "ci_high"])
    else:
        ci.columns = ["ci_low", "ci_high"]
    return pd.DataFrame({
        "variable": params.index,
        "coef": params.values,
        "std_err": se.values,
        "t_value": tvals.values,
        "p_value": pvals.values,
        "ci_low": ci["ci_low"].values,
        "ci_high": ci["ci_high"].values,
    })


def get_x_group_lookup(prep):
    lookup = {}
    order_lookup = {}
    group_order = 1
    rules = prep.get("x_rules", {})
    for group_name, cols in X_GROUPS.items():
        for within_order, col in enumerate(cols):
            model_col = transformed_name(col, rules.get(col, "raw"))
            lookup[col] = group_name
            lookup[model_col] = group_name
            order_lookup[col] = (group_order, within_order)
            order_lookup[model_col] = (group_order, within_order)
        group_order += 1
    for i, col in enumerate(X_EXTRA):
        model_col = transformed_name(col, rules.get(col, "raw"))
        lookup[col] = "x_extra"
        lookup[model_col] = "x_extra"
        order_lookup[col] = (group_order, i)
        order_lookup[model_col] = (group_order, i)
    return lookup, order_lookup


def infer_base_variable_from_design_col(var_name, x_group_lookup):
    if var_name == "const":
        return "const"
    if var_name in x_group_lookup:
        return var_name
    for base_var in x_group_lookup.keys():
        prefixes = [base_var + "_", base_var + "[", "C(" + base_var + ")", "C(" + base_var + ")["]
        for prefix in prefixes:
            if str(var_name).startswith(prefix):
                return base_var
    return var_name


def add_coef_group_order(coef, prep):
    coef = coef.copy()
    x_group_lookup, x_order_lookup = get_x_group_lookup(prep)
    group_names = []
    base_vars = []
    group_orders = []
    within_orders = []
    for var in coef["variable"].astype(str).values:
        if var == "const":
            base_var = "const"
            group_name = "constant"
            group_order = 0
            within_order = 0
        elif "_x_" in str(var):
            base_var = var
            group_name = "interactions"
            group_order = 90
            within_order = 0
        else:
            base_var = infer_base_variable_from_design_col(var, x_group_lookup)
            group_name = x_group_lookup.get(base_var, "other")
            group_order, within_order = x_order_lookup.get(base_var, (999, 999))
        base_vars.append(base_var)
        group_names.append(group_name)
        group_orders.append(group_order)
        within_orders.append(within_order)
    coef["variable_base"] = base_vars
    coef["variable_group"] = group_names
    coef["_group_order"] = group_orders
    coef["_within_group_order"] = within_orders
    coef = coef.sort_values(["_group_order", "_within_group_order", "variable"], ascending=[True, True, True]).reset_index(drop=True)
    return coef


def reorder_coef_columns(coef):
    front_cols = [
        "variable_group",
        "variable",
        "variable_base",
        "coef",
        "std_err",
        "t_value",
        "p_value",
        "ci_low",
        "ci_high",
        "y_col",
        "x_standardized_for_estimation",
    ]
    front_cols = [c for c in front_cols if c in coef.columns]
    rest_cols = [c for c in coef.columns if c not in front_cols and not c.startswith("_")]
    return coef[front_cols + rest_cols].copy()


def plot_ols_diagnostics(res, outdir, stem):
    ensure_dir(outdir)
    resid = pd.Series(np.asarray(res.resid)).astype(float)
    fitted = pd.Series(np.asarray(res.fittedvalues)).astype(float)
    if len(resid) > PLOT_SAMPLE_N:
        idx = np.random.RandomState(RANDOM_SEED).choice(len(resid), size=PLOT_SAMPLE_N, replace=False)
        resid_plot = resid.iloc[idx]
        fitted_plot = fitted.iloc[idx]
    else:
        resid_plot = resid
        fitted_plot = fitted
    resid_std = (resid_plot - resid_plot.mean()) / (resid_plot.std(ddof=0) + 1e-12)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(fitted_plot, resid_std, s=5, alpha=0.25)
    ax.axhline(0)
    ax.set_xlabel("fitted")
    ax.set_ylabel("standardized residual")
    ax.set_title("Residuals vs fitted")
    fig.tight_layout()
    fig.savefig(Path(outdir) / (stem + "__resid_vs_fitted.png"), dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(resid_std, bins=60)
    ax.set_xlabel("standardized residual")
    ax.set_ylabel("count")
    ax.set_title("Residual histogram")
    fig.tight_layout()
    fig.savefig(Path(outdir) / (stem + "__resid_hist.png"), dpi=220)
    plt.close(fig)

    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111)
    qq = resid_std.sample(50000, random_state=RANDOM_SEED) if len(resid_std) > 50000 else resid_std
    st.probplot(qq, dist="norm", plot=ax)
    ax.set_title("Residual QQ plot")
    fig.tight_layout()
    fig.savefig(Path(outdir) / (stem + "__resid_qq.png"), dpi=220)
    plt.close(fig)

    return pd.DataFrame({
        "metric": ["n", "resid_mean", "resid_std", "resid_min", "resid_max"],
        "value": [len(resid), resid.mean(), resid.std(ddof=0), resid.min(), resid.max()],
    })


def run_ols(prep):
    df = prep["df_model"].copy()
    y_col = prep["y_model"]
    x_cols = prep["x_model"]
    cat_cols = prep["cat_cols"]
    y = pd.to_numeric(df[y_col], errors="coerce")
    preferred_cols = alias_keep_model_names(prep.get("x_rules", {}))

    X, dropped_const = build_design_matrix(
        df,
        cont_cols=x_cols,
        cat_cols=cat_cols,
        add_const=True,
        preferred_order=preferred_cols,
    )
    X, dropped_alias = drop_exact_alias_columns(X, protect_cols=["const"])

    use_idx = y.dropna().index.intersection(X.dropna().index)
    y = y.loc[use_idx]
    X = X.loc[use_idx]
    df_fit = df.loc[use_idx].copy()

    # VIF is computed with an intercept through explicit auxiliary regressions.
    # The returned table includes aux_r2 so VIF can be audited as 1 / (1 - aux_r2).
    vif_before = compute_vif(X, max_rows=VIF_MAX_ROWS)
    dropped_vif = pd.DataFrame(columns=["variable", "VIF_at_drop", "aux_r2_at_drop"])
    if AUTO_DROP_HIGH_VIF:
        X, dropped_vif = iterative_drop_high_vif(X, threshold=VIF_THRESHOLD, protected=VIF_PROTECTED)
    vif_after = compute_vif(X, max_rows=VIF_MAX_ROWS)
    X_for_vif_drivers = X
    if VIF_DRIVER_MAX_ROWS is not None and len(X_for_vif_drivers) > VIF_DRIVER_MAX_ROWS:
        X_for_vif_drivers = X_for_vif_drivers.sample(VIF_DRIVER_MAX_ROWS, random_state=RANDOM_SEED)
    vif_driver_summary, vif_driver_long, vif_high_pairs = save_vif_driver_diagnostics(X_for_vif_drivers, vif_after)

    if OLS_STANDARDIZE_X_FOR_ESTIMATION:
        scale_cols = [c for c in X.columns if c != "const"]
        X_fit, scale_df = standardize_columns(X, scale_cols)
    else:
        X_fit = X.copy()
        scale_df = pd.DataFrame(columns=["column", "mean", "std"])

    cov_type = "HC3"
    cov_kwds = None
    if CLUSTER_COL in df_fit.columns:
        groups = df_fit[CLUSTER_COL]
        if groups.nunique(dropna=True) > 1:
            cov_type = "cluster"
            cov_kwds = {"groups": groups}

    model = sm.OLS(y.astype(float), X_fit.astype(float))
    if cov_kwds is None:
        res = model.fit(cov_type=cov_type)
    else:
        res = model.fit(cov_type=cov_type, cov_kwds=cov_kwds)

    coef = coef_table_from_ols(res)
    coef["y_col"] = y_col
    coef["x_standardized_for_estimation"] = coef["variable"].isin(set(scale_df["column"])) if len(scale_df) else False
    coef = add_coef_group_order(coef, prep)
    coef = reorder_coef_columns(coef)

    fit_info = pd.DataFrame([{
        "model_name": "ols_main",
        "y_col": y_col,
        "y_base": Y_COL,
        "y_transform": Y_TRANSFORM,
        "n_obs": int(len(y)),
        "n_params": int(X_fit.shape[1]),
        "r2": float(res.rsquared),
        "adj_r2": float(res.rsquared_adj),
        "aic": float(res.aic),
        "bic": float(res.bic),
        "cov_type": cov_type,
        "cluster_col": CLUSTER_COL if cov_type == "cluster" else None,
        "condition_number": matrix_condition_number(X_fit),
        "dropped_constant_cols": json.dumps(dropped_const, ensure_ascii=False),
        "dropped_alias_cols": json.dumps(dropped_alias, ensure_ascii=False),
        "auto_dropped_high_vif_cols": json.dumps(dropped_vif.to_dict(orient="records"), ensure_ascii=False),
        "alias_keep_preference_model_cols": json.dumps(preferred_cols, ensure_ascii=False),
    }])

    write_result_table(coef, "ols/ols_coef_table.csv", index=False, rounded=True)
    write_result_table(fit_info, "ols/ols_fit_info.csv", index=False, rounded=True)
    write_result_table(vif_before, "ols/vif_before_optional_drop.csv", index=False, rounded=True)
    write_result_table(vif_after, "ols/vif_final.csv", index=False, rounded=True)
    write_result_table(dropped_vif, "ols/dropped_high_vif.csv", index=False, rounded=True)
    if len(scale_df):
        write_result_table(scale_df, "ols/x_standardization_info.csv", index=False, rounded=True)

    try:
        save_text(str(res.summary()), FULL_PRECISION_DIR / "ols" / "ols_summary.txt")
    except Exception:
        pass

    if SAVE_DIAGNOSTIC_PLOTS:
        diag = plot_ols_diagnostics(res, PLOTS_DIR / "ols", "ols_main")
        write_result_table(diag, "ols/ols_diagnostics_summary.csv", index=False, rounded=True)

    return {
        "result": res,
        "coef": coef,
        "fit_info": fit_info,
        "vif_before": vif_before,
        "vif_after": vif_after,
        "vif_driver_summary": vif_driver_summary,
        "vif_driver_long": vif_driver_long,
        "vif_high_pairwise_correlations": vif_high_pairs,
        "dropped_constant_cols": dropped_const,
        "dropped_alias_cols": dropped_alias,
        "dropped_vif": dropped_vif,
        "X": X,
        "X_fit": X_fit,
        "y": y,
        "df_fit": df_fit,
    }

# Execution is handled in the final orchestration cell.


## VIF diagnostics

VIF is now computed with an intercept through explicit auxiliary regressions. The output includes `aux_r2`, so each value can be checked as `VIF = 1 / (1 - aux_r2)`. Pairwise correlations and auxiliary-regression driver tables are saved under `diagnostics/vif_drivers/`.


## Variable drop audit

This block explains where each requested variable went. It distinguishes missing columns, protected variables, manual drops, road-class reference removal, constants in the model sample, exact linear aliases, and optional high Variance Inflation Factor drops.


In [8]:
# =========================================================
# 7. Variable drop audit
# =========================================================
def parse_json_list_cell(x):
    if pd.isna(x):
        return []
    try:
        out = json.loads(str(x))
        if isinstance(out, list):
            return out
        return []
    except Exception:
        return []


def make_variable_drop_audit(prep, ols_out, df_input):
    raw_requested = []
    for group_name, cols in X_GROUPS.items():
        for c in cols:
            raw_requested.append({"source": "X_GROUPS:%s" % group_name, "raw_variable": c})
    for c in X_EXTRA:
        raw_requested.append({"source": "X_EXTRA", "raw_variable": c})
    audit = pd.DataFrame(raw_requested).drop_duplicates("raw_variable").reset_index(drop=True)

    df_input = df_input.copy()
    df_model = prep["df_model"].copy()
    x_raw_set = set(prep["x_raw"])
    x_model_set = set(prep["x_model"])
    x_rules = prep.get("x_rules", {})

    audit["exists_in_input_table"] = audit["raw_variable"].apply(lambda c: int(c in df_input.columns))
    audit["in_prepared_x_raw"] = audit["raw_variable"].apply(lambda c: int(c in x_raw_set))
    audit["transform_rule"] = audit["raw_variable"].apply(lambda c: x_rules.get(c, "raw"))
    audit["model_variable"] = audit.apply(lambda r: transformed_name(r["raw_variable"], r["transform_rule"]), axis=1)
    audit["in_prepared_x_model"] = audit["model_variable"].apply(lambda c: int(c in x_model_set))

    rows = []
    for c in audit["raw_variable"].tolist():
        if c not in df_input.columns:
            rows.append({
                "raw_variable": c,
                "input_non_null_share": np.nan,
                "input_n_unique": 0,
                "input_share_zero": np.nan,
                "model_sample_n_unique": 0,
                "model_sample_share_zero": np.nan,
            })
            continue
        s0 = pd.to_numeric(df_input[c], errors="coerce")
        s1 = pd.to_numeric(df_model[c], errors="coerce") if c in df_model.columns else pd.Series(dtype=float)
        rows.append({
            "raw_variable": c,
            "input_non_null_share": float(s0.notna().mean()),
            "input_n_unique": int(s0.nunique(dropna=True)),
            "input_share_zero": float((s0.dropna() == 0).mean()) if s0.notna().any() else np.nan,
            "model_sample_n_unique": int(s1.nunique(dropna=True)) if len(s1) else 0,
            "model_sample_share_zero": float((s1.dropna() == 0).mean()) if len(s1.dropna()) else np.nan,
        })
    audit = audit.merge(pd.DataFrame(rows), on="raw_variable", how="left")

    dropped_const = ols_out.get("dropped_constant_cols", []) if ols_out is not None else []
    dropped_alias = ols_out.get("dropped_alias_cols", []) if ols_out is not None else []
    dropped_vif_vars = []
    if ols_out is not None and "dropped_vif" in ols_out:
        dv = ols_out["dropped_vif"]
        if len(dv) and "variable" in dv.columns:
            dropped_vif_vars = dv["variable"].tolist()

    audit["dropped_as_constant_in_design"] = audit["model_variable"].apply(lambda c: int(c in dropped_const))
    audit["dropped_as_exact_alias_in_design"] = audit["model_variable"].apply(lambda c: int(c in dropped_alias))
    audit["dropped_by_auto_vif"] = audit["model_variable"].apply(lambda c: int(c in dropped_vif_vars))

    protected = set(protected_non_x())
    x_drop = set(X_DROP)

    def infer_status(row):
        c = row["raw_variable"]
        if row["exists_in_input_table"] == 0:
            return "missing_from_input_table"
        if c in x_drop:
            return "manually_dropped_by_X_DROP"
        if c in protected:
            return "protected_non_x"
        if is_road_class_share_col(c) and c == prep.get("road_class_reference"):
            return "road_class_reference_category"
        if row["in_prepared_x_raw"] == 0:
            return "not_in_prepared_x_raw"
        if row["in_prepared_x_model"] == 0:
            return "transform_or_model_name_missing"
        if row["dropped_as_constant_in_design"] == 1:
            return "dropped_constant_after_model_sample"
        if row["dropped_as_exact_alias_in_design"] == 1:
            return "dropped_exact_linear_alias"
        if row["dropped_by_auto_vif"] == 1:
            return "dropped_high_vif"
        return "kept_in_design_matrix"

    audit["drop_status"] = audit.apply(infer_status, axis=1)
    return audit

# Execution is handled in the final orchestration cell.


## R squared diagnostics

This block checks whether low explanatory power comes from weak individual predictors, weak variable groups, the current target choice, or model specification. It reports univariate R squared, group-only R squared, incremental block R squared, and the same X matrix against alternative targets when those targets are present.


In [9]:
# =========================================================
# 8. R2 diagnostics
# =========================================================
def fit_simple_ols_r2(df, y_col, x_cols, cat_cols=None):
    cat_cols = cat_cols or []
    y = pd.to_numeric(df[y_col], errors="coerce")
    x_cols = [c for c in x_cols if c in df.columns]
    cat_cols = [c for c in cat_cols if c in df.columns]
    if len(x_cols) == 0 and len(cat_cols) == 0:
        return {"n_obs": int(y.dropna().shape[0]), "n_params": 1, "r2": 0.0, "adj_r2": 0.0}
    X, _ = build_design_matrix(df, cont_cols=x_cols, cat_cols=cat_cols, add_const=True)
    X, _ = drop_exact_alias_columns(X, protect_cols=["const"])
    use_idx = y.dropna().index.intersection(X.dropna().index)
    y_fit = y.loc[use_idx].astype(float)
    X_fit = X.loc[use_idx].astype(float)
    if len(y_fit) < 50 or X_fit.shape[1] < 2:
        return {"n_obs": int(len(y_fit)), "n_params": int(X_fit.shape[1]), "r2": np.nan, "adj_r2": np.nan}
    res = sm.OLS(y_fit, X_fit).fit()
    return {"n_obs": int(len(y_fit)), "n_params": int(X_fit.shape[1]), "r2": float(res.rsquared), "adj_r2": float(res.rsquared_adj)}


def raw_to_model_vars(raw_vars, prep):
    raw_vars = [c for c in raw_vars if c in prep["x_raw"]]
    rules = prep.get("x_rules", {})
    out = []
    for c in raw_vars:
        mc = transformed_name(c, rules.get(c, "raw"))
        if mc in prep["df_model"].columns:
            out.append(mc)
    return list(OrderedDict.fromkeys(out))


def build_group_to_model_vars(prep):
    group_rows = []
    for group_name, raw_cols in X_GROUPS.items():
        raw_present = [c for c in raw_cols if c in prep["x_raw"]]
        model_cols = raw_to_model_vars(raw_present, prep)
        group_rows.append({
            "group_name": group_name,
            "raw_vars": raw_present,
            "model_vars": model_cols,
            "n_model_vars": len(model_cols),
        })
    return group_rows


def run_r2_diagnostics(prep):
    df_r2 = prep["df_model"].copy()
    y_model = prep["y_model"]
    outdir_rel = "diagnostics/r2_diagnostics"

    univar_rows = []
    for c in prep["x_model"]:
        if c not in df_r2.columns:
            continue
        res = fit_simple_ols_r2(df_r2, y_model, [c], [])
        s = pd.to_numeric(df_r2[c], errors="coerce")
        y = pd.to_numeric(df_r2[y_model], errors="coerce")
        tmp = pd.DataFrame({"x": s, "y": y}).dropna()
        corr = np.nan
        if len(tmp) > 2 and tmp["x"].nunique() > 1 and tmp["y"].nunique() > 1:
            corr = float(tmp["x"].corr(tmp["y"]))
        univar_rows.append({
            "variable": c,
            "n_obs": res["n_obs"],
            "r2": res["r2"],
            "adj_r2": res["adj_r2"],
            "corr_with_y": corr,
            "abs_corr_with_y": abs(corr) if pd.notna(corr) else np.nan,
            "n_unique": int(s.nunique(dropna=True)),
            "share_zero": float((s.dropna() == 0).mean()) if len(s.dropna()) else np.nan,
        })
    univar_r2 = pd.DataFrame(univar_rows).sort_values("r2", ascending=False)
    write_result_table(univar_r2, outdir_rel + "/univariate_r2_by_variable.csv", index=False, rounded=True)

    group_rows = build_group_to_model_vars(prep)
    block_rows = []
    for gr in group_rows:
        res = fit_simple_ols_r2(df_r2, y_model, gr["model_vars"], [])
        block_rows.append({
            "model": "group_only",
            "group_name": gr["group_name"],
            "n_model_vars": gr["n_model_vars"],
            "n_obs": res["n_obs"],
            "n_params": res["n_params"],
            "r2": res["r2"],
            "adj_r2": res["adj_r2"],
        })
    block_r2 = pd.DataFrame(block_rows)
    write_result_table(block_r2, outdir_rel + "/group_only_r2.csv", index=False, rounded=True)

    cumulative_vars = []
    increment_rows = []
    prev_r2 = 0.0
    for gr in group_rows:
        cumulative_vars = list(OrderedDict.fromkeys(cumulative_vars + gr["model_vars"]))
        res = fit_simple_ols_r2(df_r2, y_model, cumulative_vars, [])
        cur_r2 = res["r2"]
        increment_rows.append({
            "added_group": gr["group_name"],
            "n_cumulative_vars": len(cumulative_vars),
            "n_obs": res["n_obs"],
            "n_params": res["n_params"],
            "r2": cur_r2,
            "adj_r2": res["adj_r2"],
            "incremental_r2": cur_r2 - prev_r2 if pd.notna(cur_r2) else np.nan,
        })
        if pd.notna(cur_r2):
            prev_r2 = cur_r2
    incremental_r2 = pd.DataFrame(increment_rows)
    write_result_table(incremental_r2, outdir_rel + "/incremental_block_r2.csv", index=False, rounded=True)

    alt_targets = []
    for target in ["speed_kmh", "duration", "final_distance_m", "overspeed_20"]:
        if target in df_r2.columns:
            res = fit_simple_ols_r2(df_r2, target, prep["x_model"], [])
            alt_targets.append({
                "target": target,
                "n_obs": res["n_obs"],
                "n_params": res["n_params"],
                "r2": res["r2"],
                "adj_r2": res["adj_r2"],
            })
    alt_target_r2 = pd.DataFrame(alt_targets)
    write_result_table(alt_target_r2, outdir_rel + "/same_x_different_targets_r2.csv", index=False, rounded=True)

    print("Top univariate R2 variables:")
    display(round_numeric_columns(univar_r2.head(30), 3))
    print("Group-only R2:")
    display(round_numeric_columns(block_r2.sort_values("r2", ascending=False), 3))
    print("Incremental block R2:")
    display(round_numeric_columns(incremental_r2, 3))
    print("Same X, different target R2:")
    display(round_numeric_columns(alt_target_r2, 3))

    return {
        "univariate_r2": univar_r2,
        "group_only_r2": block_r2,
        "incremental_r2": incremental_r2,
        "alt_target_r2": alt_target_r2,
    }

# Execution is handled in the final orchestration cell.


## XGBoost and SHAP

This section uses the same selected variables and transformations as OLS. XGBoost can capture nonlinear effects and interactions. SHAP then helps inspect which variables drive predictions and where nonlinear thresholds appear.


In [10]:
# =========================================================
# 9. XGBoost and SHAP functions
# =========================================================
def import_ml_dependencies():
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        try:
            from sklearn.metrics import root_mean_squared_error
        except Exception:
            root_mean_squared_error = None
        import xgboost as xgb
        import shap
        return {
            "ok": True,
            "train_test_split": train_test_split,
            "mean_absolute_error": mean_absolute_error,
            "mean_squared_error": mean_squared_error,
            "root_mean_squared_error": root_mean_squared_error,
            "r2_score": r2_score,
            "xgb": xgb,
            "shap": shap,
            "error": None,
        }
    except Exception as exc:
        return {"ok": False, "error": repr(exc)}


def ml_rmse(y_true, y_pred, deps):
    """Version-safe root mean squared error.

    Newer scikit-learn versions expose root_mean_squared_error directly.
    Older versions can compute the same value as sqrt(mean_squared_error).
    """
    rmse_func = deps.get("root_mean_squared_error")
    if rmse_func is not None:
        return float(rmse_func(y_true, y_pred))
    mse_func = deps["mean_squared_error"]
    return float(np.sqrt(mse_func(y_true, y_pred)))


def build_ml_matrix(prep):
    df = prep["df_model"].copy()
    y_col = prep["y_model"]
    x_cols = prep["x_model"]
    cat_cols = prep["cat_cols"]
    y = pd.to_numeric(df[y_col], errors="coerce")
    X, _ = build_design_matrix(df, cont_cols=x_cols, cat_cols=cat_cols, add_const=False, preferred_order=alias_keep_model_names(prep.get("x_rules", {})))
    X = X.replace([np.inf, -np.inf], np.nan)
    use_idx = y.dropna().index.intersection(X.dropna().index)
    X = X.loc[use_idx].astype(float)
    y = y.loc[use_idx].astype(float)
    df_fit = df.loc[use_idx].copy()
    if len(X) > XGB_MAX_ROWS:
        sample_idx = X.sample(XGB_MAX_ROWS, random_state=RANDOM_SEED).index
        X = X.loc[sample_idx]
        y = y.loc[sample_idx]
        df_fit = df_fit.loc[sample_idx]
    return X, y, df_fit


def run_xgboost_shap(prep):
    deps = import_ml_dependencies()
    outdir = FULL_PRECISION_DIR / "xgboost_shap"
    rounded_outdir = ROUNDED_DIR / "xgboost_shap"
    plotdir = PLOTS_DIR / "xgboost_shap"
    ensure_dir(outdir)
    ensure_dir(rounded_outdir)
    ensure_dir(plotdir)
    if not deps["ok"]:
        msg = "Skipped XGBoost SHAP because dependencies failed to import: %s" % deps["error"]
        print(msg)
        save_text(msg, outdir / "xgboost_shap_skipped.txt")
        return None

    X, y, df_fit = build_ml_matrix(prep)
    train_test_split = deps["train_test_split"]
    mean_absolute_error = deps["mean_absolute_error"]
    r2_score = deps["r2_score"]
    xgb = deps["xgb"]
    shap = deps["shap"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=XGB_TEST_SIZE, random_state=RANDOM_SEED
    )

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train)

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    metrics = pd.DataFrame([
        {
            "split": "train",
            "n": int(len(y_train)),
            "mae": float(mean_absolute_error(y_train, pred_train)),
            "rmse": ml_rmse(y_train, pred_train, deps),
            "r2": float(r2_score(y_train, pred_train)),
        },
        {
            "split": "test",
            "n": int(len(y_test)),
            "mae": float(mean_absolute_error(y_test, pred_test)),
            "rmse": ml_rmse(y_test, pred_test, deps),
            "r2": float(r2_score(y_test, pred_test)),
        },
    ])
    write_result_table(metrics, "xgboost_shap/xgb_metrics.csv", index=False, rounded=True)
    display(round_numeric_columns(metrics, 3))

    importance = pd.DataFrame({
        "variable": X.columns,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    write_result_table(importance, "xgboost_shap/xgb_feature_importance.csv", index=False, rounded=True)

    if len(X_test) > SHAP_SAMPLE_N:
        X_shap = X_test.sample(SHAP_SAMPLE_N, random_state=RANDOM_SEED)
    else:
        X_shap = X_test.copy()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_importance = pd.DataFrame({
        "variable": X_shap.columns,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    write_result_table(shap_importance, "xgboost_shap/shap_mean_abs_importance.csv", index=False, rounded=True)

    plt.figure(figsize=(10, max(6, 0.35 * min(SHAP_TOP_N, X_shap.shape[1]))))
    shap.summary_plot(shap_values, X_shap, max_display=SHAP_TOP_N, show=False)
    plt.tight_layout()
    plt.savefig(plotdir / "shap_summary_beeswarm.png", dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, max(6, 0.35 * min(SHAP_TOP_N, X_shap.shape[1]))))
    shap.summary_plot(shap_values, X_shap, plot_type="bar", max_display=SHAP_TOP_N, show=False)
    plt.tight_layout()
    plt.savefig(plotdir / "shap_summary_bar.png", dpi=220, bbox_inches="tight")
    plt.close()

    top_vars = shap_importance["variable"].head(min(8, len(shap_importance))).tolist()
    for v in top_vars:
        try:
            shap.dependence_plot(v, shap_values, X_shap, show=False)
            plt.tight_layout()
            safe_v = re.sub(r"[^A-Za-z0-9_]+", "_", str(v))[:120]
            plt.savefig(plotdir / ("shap_dependence__%s.png" % safe_v), dpi=220, bbox_inches="tight")
            plt.close()
        except Exception as exc:
            print("Could not plot SHAP dependence for %s: %s" % (v, exc))

    return {
        "model": model,
        "metrics": metrics,
        "importance": importance,
        "shap_importance": shap_importance,
        "X_test": X_test,
        "y_test": y_test,
    }

# Execution is handled in the final orchestration cell.


In [11]:
# =========================================================
# 10. Run model specifications and save effective configuration
# =========================================================
def format_python_list(name, values, indent="    "):
    lines = [name + " = ["]
    for v in values:
        lines.append(indent + repr(str(v)) + ",")
    lines.append("]")
    return "\n".join(lines)


def save_effective_configuration(prep, run_name, row_query):
    effective_config = {
        "run_name": run_name,
        "trip_table_path": str(TRIP_TABLE_PATH),
        "base_output_root": str(BASE_OUTPUT_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "full_precision_dir": str(FULL_PRECISION_DIR),
        "rounded_dir": str(ROUNDED_DIR),
        "y_col": Y_COL,
        "y_transform": Y_TRANSFORM,
        "row_query": row_query,
        "cluster_col": CLUSTER_COL,
        "run_multiple_model_specs": RUN_MULTIPLE_MODEL_SPECS,
        "primary_run_name": PRIMARY_RUN_NAME,
        "model_run_specs": dict(MODEL_RUN_SPECS),
        "x_groups": {k: list(v) for k, v in X_GROUPS.items()},
        "x_extra": X_EXTRA,
        "x_drop": X_DROP,
        "categorical_vars": CATEGORICAL_VARS,
        "derived_variable_specs": DERIVED_VARIABLE_SPECS,
        "add_interactions": ADD_INTERACTIONS,
        "interaction_specs": INTERACTION_SPECS,
        "drop_mechanical_x_when_y_is_speed": DROP_MECHANICAL_X_WHEN_Y_IS_SPEED,
        "mechanical_x_when_y_speed": MECHANICAL_X_WHEN_Y_SPEED,
        "drop_one_road_class_reference": DROP_ONE_ROAD_CLASS_REFERENCE,
        "road_class_reference_mode": ROAD_CLASS_REFERENCE_MODE,
        "road_class_reference_used": prep.get("road_class_reference"),
        "variable_transforms": dict(VARIABLE_TRANSFORMS),
        "pattern_variable_transforms": dict(PATTERN_VARIABLE_TRANSFORMS),
        "auto_apply_recommended_transforms": AUTO_APPLY_RECOMMENDED_TRANSFORMS,
        "alias_keep_preference_raw": ALIAS_KEEP_PREFERENCE_RAW,
        "auto_drop_high_vif": AUTO_DROP_HIGH_VIF,
        "vif_threshold": VIF_THRESHOLD,
        "vif_max_rows": VIF_MAX_ROWS,
        "vif_driver_max_rows": VIF_DRIVER_MAX_ROWS,
        "run_ols": RUN_OLS,
        "run_xgboost_shap": RUN_XGBOOST_SHAP,
        "xgb_params": XGB_PARAMS,
    }
    save_json(effective_config, OUTPUT_ROOT / "effective_config.json")

    copy_text = []
    copy_text.append("Y_COL = %r" % Y_COL)
    copy_text.append("Y_TRANSFORM = %r" % Y_TRANSFORM)
    copy_text.append("ROW_QUERY = %r" % row_query)
    copy_text.append("")
    copy_text.append(format_python_list("X_SELECTED_RAW", prep["x_raw"]))
    copy_text.append("")
    copy_text.append(format_python_list("X_SELECTED_MODEL", prep["x_model"]))
    save_text("\n".join(copy_text), DIAGNOSTICS_DIR / "final_selected_x_y_block.py")


def run_one_model_spec(run_name, row_query, run_dashboard=False, run_r2=False, run_xgb=False):
    set_output_dirs_for_run(run_name)
    print("")
    print("=" * 80)
    print("Running model spec:", run_name)
    print("Row query:", row_query)
    print("Output root:", OUTPUT_ROOT)

    trip_df_local = load_and_filter_trip_table(row_query=row_query)
    prep = prepare_modeling_data(trip_df_local)
    save_preparation_outputs(prep)

    if run_dashboard:
        variable_distribution_summary_local = make_variable_distribution_dashboard(prep)
    else:
        variable_distribution_summary_local = None
        print("Skipped distribution dashboard for this run.")

    if RUN_OLS:
        ols_out = run_ols(prep)
        display(round_numeric_columns(ols_out["fit_info"], 3))
        display(round_numeric_columns(ols_out["coef"].head(50), 3))
        print("Correct VIF table, computed with intercept:")
        display(round_numeric_columns(ols_out["vif_after"], 3))
    else:
        ols_out = None
        print("RUN_OLS is False. Skipped OLS.")

    variable_drop_audit_local = make_variable_drop_audit(prep, ols_out, trip_df_local)
    write_result_table(variable_drop_audit_local, "diagnostics/variable_drop_audit.csv", index=False, rounded=True)
    print("Variable drop audit saved.")

    if run_r2:
        r2_diagnostics_local = run_r2_diagnostics(prep)
    else:
        r2_diagnostics_local = None
        print("Skipped R2 diagnostics for this run.")

    if run_xgb:
        xgb_shap_results_local = run_xgboost_shap(prep)
    else:
        xgb_shap_results_local = None
        print("Skipped XGBoost and SHAP for this run.")

    save_effective_configuration(prep, run_name, row_query)

    fit_info = pd.DataFrame()
    if ols_out is not None:
        fit_info = ols_out["fit_info"].copy()
        fit_info.insert(0, "run_name", run_name)
        fit_info.insert(1, "row_query", row_query)
        fit_info["n_raw_x"] = len(prep["x_raw"])
        fit_info["n_model_x_before_dummies"] = len(prep["x_model"])
        fit_info["n_interactions_created"] = int((prep["interaction_info"].get("status", pd.Series(dtype=str)) == "created").sum()) if len(prep["interaction_info"]) else 0

    print("Done with run:", run_name)
    print("Full precision outputs:", FULL_PRECISION_DIR)
    print("Rounded outputs:", ROUNDED_DIR)
    print("Diagnostics:", DIAGNOSTICS_DIR)
    print("Plots:", PLOTS_DIR)

    return {
        "run_name": run_name,
        "row_query": row_query,
        "trip_df": trip_df_local,
        "prepared": prep,
        "variable_distribution_summary": variable_distribution_summary_local,
        "ols": ols_out,
        "variable_drop_audit": variable_drop_audit_local,
        "r2_diagnostics": r2_diagnostics_local,
        "xgb_shap": xgb_shap_results_local,
        "fit_info": fit_info,
    }


def write_model_comparison(run_outputs):
    fit_rows = []
    for out in run_outputs.values():
        fit = out.get("fit_info")
        if fit is not None and len(fit) > 0:
            fit_rows.append(fit)

    if len(fit_rows) == 0:
        print("No OLS fit rows to compare.")
        return pd.DataFrame()

    model_comparison = pd.concat(fit_rows, ignore_index=True)
    ensure_dir(BASE_OUTPUT_ROOT)
    write_csv_atomic(model_comparison, BASE_OUTPUT_ROOT / "model_comparison_full_precision.csv", index=False)
    write_csv_atomic(round_numeric_columns(model_comparison, 3), BASE_OUTPUT_ROOT / "model_comparison.csv", index=False)
    print("Model comparison saved:", BASE_OUTPUT_ROOT / "model_comparison.csv")
    display(round_numeric_columns(model_comparison, 3))
    return model_comparison


if RUN_MULTIPLE_MODEL_SPECS:
    run_specs = MODEL_RUN_SPECS
else:
    run_specs = OrderedDict([("single_run", ROW_QUERY)])

if PRIMARY_RUN_NAME not in run_specs:
    PRIMARY_RUN_NAME_EFFECTIVE = next(iter(run_specs.keys()))
else:
    PRIMARY_RUN_NAME_EFFECTIVE = PRIMARY_RUN_NAME

run_outputs = OrderedDict()
for run_name, row_query in run_specs.items():
    is_primary = run_name == PRIMARY_RUN_NAME_EFFECTIVE
    run_outputs[run_name] = run_one_model_spec(
        run_name=run_name,
        row_query=row_query,
        run_dashboard=RUN_VARIABLE_DISTRIBUTION_DASHBOARD and (RUN_DISTRIBUTION_DASHBOARD_FOR_ALL_SPECS or is_primary),
        run_r2=RUN_R2_DIAGNOSTICS and (RUN_R2_DIAGNOSTICS_FOR_ALL_SPECS or is_primary),
        run_xgb=RUN_XGBOOST_SHAP and (RUN_XGBOOST_SHAP_FOR_ALL_SPECS or is_primary),
    )

model_comparison = write_model_comparison(run_outputs)

# Expose the primary run under the old variable names for compatibility with older ad hoc cells.
primary_output = run_outputs[PRIMARY_RUN_NAME_EFFECTIVE]
trip_df = primary_output["trip_df"]
prepared = primary_output["prepared"]
variable_distribution_summary = primary_output["variable_distribution_summary"]
ols_results = primary_output["ols"]
variable_drop_audit = primary_output["variable_drop_audit"]
r2_diagnostics = primary_output["r2_diagnostics"]
xgb_shap_results = primary_output["xgb_shap"]

set_output_dirs_for_run(PRIMARY_RUN_NAME_EFFECTIVE)
print("")
print("Primary run exposed as legacy variables:", PRIMARY_RUN_NAME_EFFECTIVE)
print("Selected raw X count:", len(prepared["x_raw"]))
print("Selected model X count:", len(prepared["x_model"]))



Running model spec: h00_04
Row query: speed_kmh <= 40 and start_hour_local >= 0 and start_hour_local < 4
Output root: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/h00_04
Raw modeling table shape: (451887, 146)
Action context columns:
segment_task_orientation  segment_grab_context  
to_deliver                no_grab_endpoint          279267
to_fetch                  post_grab_after_update     72427
                          no_grab_endpoint           52845
to_deliver                pre_grab_interrupted       22563
                          post_grab_after_update     15773
to_fetch                  pre_grab_interrupted        9012
After row query: 451887 -> 12160
Dropped road-class reference column: road_class_share_levelFourRoad_lenw_mean
Rows after dropping model missing values: 12160 -> 12160
Selected raw X count: 16
Selected model X count: 16
Road-class reference column: road_class_share_levelFourRoad_lenw_mean


,variable,manual_rule,recommended_rule,recommend_reason,final_rule_if_applied
0,onhand_order_count_start,raw,raw,low_cardinality,raw
1,time_pressure_min,raw,signed_log1p,has_negative_values_and_high_skew,raw
2,is_weekend_local,raw,raw,binary_or_constant,raw
3,rider_avg_orders_per_active_day,raw,raw,no_strong_transform_signal,raw
4,full_time,raw,raw,binary_or_constant,raw
5,rider_mean_segment_distance_m,div:1000,log,positive_and_high_skew,div:1000
6,road_class_share_levelThreeRoad_lenw_mean,raw,raw,no_strong_transform_signal,raw
7,road_class_share_secondaryRoad_lenw_mean,raw,raw,no_strong_transform_signal,raw
8,road_class_share_nationalRoad_lenw_mean,raw,log1p,nonnegative_with_zero_and_high_skew,raw
9,road_class_share_provincialRoad_lenw_mean,raw,log1p,nonnegative_with_zero_and_high_skew,raw


Skipped distribution dashboard for this run.


,model_name,y_col,y_base,y_transform,n_obs,n_params,r2,adj_r2,aic,bic,cov_type,cluster_col,condition_number,dropped_constant_cols,dropped_alias_cols,auto_dropped_high_vif_cols,alias_keep_preference_model_cols
0,ols_main,speed_kmh,speed_kmh,raw,12160,18,0.205,0.204,75854.332,75987.638,cluster,courier_id,2175.547,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau..."


,variable_group,variable,variable_base,coef,std_err,t_value,p_value,ci_low,ci_high,y_col,x_standardized_for_estimation
0,constant,const,const,11.253,0.822,13.687,0.000,9.641,12.864,speed_kmh,False
1,trip_state,onhand_order_count_start,onhand_order_count_start,0.566,0.080,7.110,0.000,0.410,0.722,speed_kmh,False
2,trip_state,time_pressure_min,time_pressure_min,0.012,0.006,2.035,0.042,0.000,0.024,speed_kmh,False
3,trip_state,is_weekend_local,is_weekend_local,-0.077,0.127,-0.604,0.546,-0.326,0.172,speed_kmh,False
4,rider_behavior,rider_avg_orders_per_active_day,rider_avg_orders_per_active_day,0.032,0.010,3.262,0.001,0.013,0.051,speed_kmh,False
5,rider_behavior,full_time,full_time,-0.004,0.175,-0.025,0.980,-0.347,0.338,speed_kmh,False
6,rider_behavior,div1000_rider_mean_segment_distance_m,div1000_rider_mean_segment_distance_m,2.098,0.172,12.188,0.000,1.761,2.435,speed_kmh,False
7,road_class_shares,road_class_share_levelThreeRoad_lenw_mean,road_class_share_levelThreeRoad_lenw_mean,0.938,0.265,3.536,0.000,0.418,1.458,speed_kmh,False
8,road_class_shares,road_class_share_secondaryRoad_lenw_mean,road_class_share_secondaryRoad_lenw_mean,3.276,0.312,10.492,0.000,2.664,3.888,speed_kmh,False
9,road_class_shares,road_class_share_nationalRoad_lenw_mean,road_class_share_nationalRoad_lenw_mean,1.741,0.882,1.973,0.048,0.012,3.469,speed_kmh,False


...

Correct VIF table, computed with intercept:


,variable,VIF,aux_r2,n_obs,n_predictors_in_aux,status
0,ln1p_poi_count_30m_lenw_mean,3.756,0.734,71767,16,ok
1,poi_mix_entropy_norm_30m_lenw_mean,3.253,0.693,71767,16,ok
2,road_class_share_secondaryRoad_lenw_mean,2.361,0.576,71767,16,ok
3,road_class_share_levelThreeRoad_lenw_mean,2.217,0.549,71767,16,ok
4,closeness_per_km_z_mean_end_lenw_mean,2.163,0.538,71767,16,ok
5,betweenness_log1p_z_mean_end_lenw_mean,1.859,0.462,71767,16,ok
6,rider_avg_orders_per_active_day,1.668,0.400,71767,16,ok
7,div1000_rider_mean_segment_distance_m,1.601,0.375,71767,16,ok
8,time_pressure_min,1.491,0.329,71767,16,ok
9,road_class_share_provincialRoad_lenw_mean,1.459,0.315,71767,16,ok


Variable drop audit saved.
Skipped R2 diagnostics for this run.
Skipped XGBoost and SHAP for this run.
Done with run: h20_24
Full precision outputs: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/h20_24/full_precision
Rounded outputs: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/h20_24/rounded
Diagnostics: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/h20_24/diagnostics
Plots: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/h20_24/plots
Model comparison saved: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/model_comparison.csv


,run_name,row_query,model_name,y_col,y_base,y_transform,n_obs,n_params,r2,adj_r2,...,cov_type,cluster_col,condition_number,dropped_constant_cols,dropped_alias_cols,auto_dropped_high_vif_cols,alias_keep_preference_model_cols,n_raw_x,n_model_x_before_dummies,n_interactions_created
0,h00_04,speed_kmh <= 40 and start_hour_local >= 0 and ...,ols_main,speed_kmh,speed_kmh,raw,12160,18,0.205,0.204,...,cluster,courier_id,2175.547,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
1,h04_08,speed_kmh <= 40 and start_hour_local >= 4 and ...,ols_main,speed_kmh,speed_kmh,raw,10608,18,0.217,0.216,...,cluster,courier_id,2384.806,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
2,h08_12,speed_kmh <= 40 and start_hour_local >= 8 and ...,ols_main,speed_kmh,speed_kmh,raw,89140,18,0.198,0.198,...,cluster,courier_id,2120.727,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
3,h12_16,speed_kmh <= 40 and start_hour_local >= 12 and...,ols_main,speed_kmh,speed_kmh,raw,119020,18,0.212,0.212,...,cluster,courier_id,2135.570,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
4,h16_20,speed_kmh <= 40 and start_hour_local >= 16 and...,ols_main,speed_kmh,speed_kmh,raw,149154,18,0.189,0.189,...,cluster,courier_id,2136.903,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
5,h20_24,speed_kmh <= 40 and start_hour_local >= 20 and...,ols_main,speed_kmh,speed_kmh,raw,71767,18,0.207,0.207,...,cluster,courier_id,2209.659,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0



Primary run exposed as legacy variables: h00_04
Selected raw X count: 16
Selected model X count: 16


## Time heterogeneity coefficient comparison

This block combines the six four-hour OLS outputs into long and wide coefficient tables.


In [14]:
# =========================================================
# 11. Time heterogeneity coefficient comparison table
#     Six four-hour models side by side
#     Python 3.8 compatible
#     Fixed pandas fillna ndarray issue
# =========================================================

from pathlib import Path
import os
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 11.0 Output paths
# ---------------------------------------------------------

TIME_HET_EXPORT_DIR = BASE_OUTPUT_ROOT / "coef_comparison"
TIME_HET_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TIME_HET_COEF_LONG_CSV = TIME_HET_EXPORT_DIR / "time_heterogeneity_coef_long.csv"
TIME_HET_COEF_WIDE_CSV = TIME_HET_EXPORT_DIR / "time_heterogeneity_coef_wide.csv"
TIME_HET_COEF_WIDE_PRETTY_CSV = TIME_HET_EXPORT_DIR / "time_heterogeneity_coef_wide_pretty.csv"
TIME_HET_FIT_COMPARISON_CSV = TIME_HET_EXPORT_DIR / "time_heterogeneity_fit_comparison.csv"
TIME_HET_VARIABLE_PRESENCE_CSV = TIME_HET_EXPORT_DIR / "time_heterogeneity_variable_presence.csv"


# ---------------------------------------------------------
# 11.1 Helpers
# ---------------------------------------------------------

def th_safe_write_csv(df_out, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df_out.to_csv(tmp, index=index)
    os.replace(str(tmp), str(path))


def th_p_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.1:
        return "+"
    return ""


def th_window_info(run_name):
    if "TIME_HETEROGENEITY_WINDOWS" in globals() and run_name in TIME_HETEROGENEITY_WINDOWS:
        info = TIME_HETEROGENEITY_WINDOWS[run_name]
        return info.get("label", run_name), info.get("hour_start", np.nan), info.get("hour_end", np.nan)
    return run_name, np.nan, np.nan


def th_get_var_col(coef):
    if "variable" in coef.columns:
        return "variable"
    if "original_variable" in coef.columns:
        return "original_variable"
    if "term" in coef.columns:
        return "term"
    raise ValueError("Cannot find variable column in coefficient table.")


def th_ensure_coef_schema(coef):
    out = coef.copy()

    var_col = th_get_var_col(out)
    if var_col != "variable":
        out["variable"] = out[var_col].astype(str)

    if "variable_base" not in out.columns:
        out["variable_base"] = out["variable"].astype(str)

    if "variable_group" not in out.columns:
        out["variable_group"] = "unknown"

    required_numeric = ["coef", "std_err", "t_value", "p_value", "ci_low", "ci_high"]
    for c in required_numeric:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")

    return out


def th_collect_coef_long(run_outputs):
    rows = []

    for run_name, out in run_outputs.items():
        ols = out.get("ols")
        if ols is None or ols.get("coef") is None:
            continue

        label, h0, h1 = th_window_info(run_name)

        coef = th_ensure_coef_schema(ols["coef"])
        
        for c in ["run_name", "time_window", "hour_start", "hour_end"]:
            if c in coef.columns:
                coef = coef.drop(columns=[c])

        coef.insert(0, "run_name", run_name)
        coef.insert(1, "time_window", label)
        coef.insert(2, "hour_start", h0)
        coef.insert(3, "hour_end", h1)

        fit = out.get("fit_info", pd.DataFrame())
        if isinstance(fit, pd.DataFrame) and len(fit) > 0:
            coef["n_obs"] = pd.to_numeric(fit.iloc[0].get("n_obs", np.nan), errors="coerce")
            coef["r2"] = pd.to_numeric(fit.iloc[0].get("r2", np.nan), errors="coerce")
            coef["adj_r2"] = pd.to_numeric(fit.iloc[0].get("adj_r2", np.nan), errors="coerce")

        rows.append(coef)

    if len(rows) == 0:
        return pd.DataFrame()

    long = pd.concat(rows, ignore_index=True)

    long["sig"] = long["p_value"].apply(th_p_stars)
    long["coef_rounded"] = long["coef"].round(4)
    long["std_err_rounded"] = long["std_err"].round(4)

    long["coef_with_sig"] = long["coef_rounded"].apply(
        lambda x: "" if pd.isna(x) else "%.4f" % float(x)
    ) + long["sig"]

    return long


def th_get_window_order(long):
    if "TIME_HETEROGENEITY_WINDOWS" in globals():
        labels = []
        for run_name in TIME_HETEROGENEITY_WINDOWS.keys():
            label, _, _ = th_window_info(run_name)
            labels.append(label)
        labels = [x for x in labels if x in set(long["time_window"])]
        if len(labels) > 0:
            return labels

    tmp = (
        long[["time_window", "hour_start"]]
        .drop_duplicates()
        .sort_values(["hour_start", "time_window"])
    )
    return tmp["time_window"].tolist()


def th_build_variable_order(long):
    # Prefer the first time window's order. Then append variables that appear only later.
    window_order = th_get_window_order(long)
    first_window = window_order[0] if len(window_order) > 0 else long["time_window"].iloc[0]

    order_source = (
        long[long["time_window"] == first_window][["variable_group", "variable", "variable_base"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    order_source["_var_order"] = np.arange(len(order_source), dtype=float)

    all_vars = (
        long[["variable_group", "variable", "variable_base"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    all_vars = all_vars.merge(
        order_source[["variable", "_var_order"]],
        on="variable",
        how="left",
    )

    # pandas 2.x does not allow fillna with ndarray.
    missing_mask = all_vars["_var_order"].isna()
    if missing_mask.any():
        start_order = float(len(order_source))
        all_vars.loc[missing_mask, "_var_order"] = (
            start_order + np.arange(int(missing_mask.sum()), dtype=float)
        )

    all_vars = all_vars.sort_values(["_var_order", "variable"]).reset_index(drop=True)
    return all_vars, window_order


def th_pivot_metric(long, metric, window_order):
    piv = long.pivot_table(
        index="variable",
        columns="time_window",
        values=metric,
        aggfunc="first",
    )

    # Reorder windows.
    ordered_cols = [c for c in window_order if c in piv.columns]
    remaining_cols = [c for c in piv.columns if c not in set(ordered_cols)]
    piv = piv[ordered_cols + remaining_cols]

    piv.columns = ["%s__%s" % (metric, c) for c in piv.columns]
    return piv.reset_index()


def th_build_wide_tables(long):
    if len(long) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    all_vars, window_order = th_build_variable_order(long)

    metric_frames = []
    for metric in ["coef", "std_err", "t_value", "p_value", "ci_low", "ci_high"]:
        metric_frames.append(th_pivot_metric(long, metric, window_order))

    wide = all_vars.copy()
    for frame in metric_frames:
        wide = wide.merge(frame, on="variable", how="left")

    pretty_piv = long.pivot_table(
        index="variable",
        columns="time_window",
        values="coef_with_sig",
        aggfunc="first",
    )

    ordered_cols = [c for c in window_order if c in pretty_piv.columns]
    remaining_cols = [c for c in pretty_piv.columns if c not in set(ordered_cols)]
    pretty_piv = pretty_piv[ordered_cols + remaining_cols]

    pretty = pretty_piv.reset_index()
    pretty = all_vars.merge(pretty, on="variable", how="left")

    presence_piv = long.pivot_table(
        index="variable",
        columns="time_window",
        values="coef",
        aggfunc=lambda x: int(pd.notna(x).any()),
        fill_value=0,
    )

    ordered_cols = [c for c in window_order if c in presence_piv.columns]
    remaining_cols = [c for c in presence_piv.columns if c not in set(ordered_cols)]
    presence_piv = presence_piv[ordered_cols + remaining_cols]

    presence = presence_piv.reset_index()
    presence = all_vars.merge(presence, on="variable", how="left")

    wide = wide.drop(columns=["_var_order"], errors="ignore")
    pretty = pretty.drop(columns=["_var_order"], errors="ignore")
    presence = presence.drop(columns=["_var_order"], errors="ignore")

    return wide, pretty, presence


def th_fit_comparison(run_outputs):
    rows = []

    for run_name, out in run_outputs.items():
        fit = out.get("fit_info", pd.DataFrame())
        if not isinstance(fit, pd.DataFrame) or len(fit) == 0:
            continue

        label, h0, h1 = th_window_info(run_name)

        one = fit.copy()

        # Avoid duplicate-column errors when fit_info already contains these columns.
        for c in ["run_name", "time_window", "hour_start", "hour_end"]:
            if c in one.columns:
                one = one.drop(columns=[c])

        one.insert(0, "run_name", run_name)
        one.insert(1, "time_window", label)
        one.insert(2, "hour_start", h0)
        one.insert(3, "hour_end", h1)

        rows.append(one)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


# ---------------------------------------------------------
# 11.2 Build and save comparison tables
# ---------------------------------------------------------

if "run_outputs" not in globals():
    raise NameError("run_outputs is missing. Run the model-spec orchestration cell first.")

th_coef_long = th_collect_coef_long(run_outputs)

if len(th_coef_long) == 0:
    raise ValueError("No OLS coefficient results found in run_outputs.")

th_coef_wide, th_coef_wide_pretty, th_presence = th_build_wide_tables(th_coef_long)
th_fit = th_fit_comparison(run_outputs)

th_safe_write_csv(th_coef_long, TIME_HET_COEF_LONG_CSV, index=False)
th_safe_write_csv(th_coef_wide, TIME_HET_COEF_WIDE_CSV, index=False)
th_safe_write_csv(th_coef_wide_pretty, TIME_HET_COEF_WIDE_PRETTY_CSV, index=False)
th_safe_write_csv(th_fit, TIME_HET_FIT_COMPARISON_CSV, index=False)
th_safe_write_csv(th_presence, TIME_HET_VARIABLE_PRESENCE_CSV, index=False)

print("\n" + "=" * 80)
print("Time heterogeneity coefficient comparison finished")
print("=" * 80)
print("Long coefficient table:", TIME_HET_COEF_LONG_CSV)
print("Wide coefficient table:", TIME_HET_COEF_WIDE_CSV)
print("Wide pretty coefficient table:", TIME_HET_COEF_WIDE_PRETTY_CSV)
print("Fit comparison:", TIME_HET_FIT_COMPARISON_CSV)
print("Variable presence table:", TIME_HET_VARIABLE_PRESENCE_CSV)

print("\nFit comparison preview:")
display(round_numeric_columns(th_fit, 4) if "round_numeric_columns" in globals() else th_fit)

print("\nWide coefficient table preview:")
display(th_coef_wide_pretty.head(80))


Time heterogeneity coefficient comparison finished
Long coefficient table: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/coef_comparison/time_heterogeneity_coef_long.csv
Wide coefficient table: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/coef_comparison/time_heterogeneity_coef_wide.csv
Wide pretty coefficient table: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/coef_comparison/time_heterogeneity_coef_wide_pretty.csv
Fit comparison: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/coef_comparison/time_heterogeneity_fit_comparison.csv
Variable presence table: /Users/kwk1001/Desktop/Meituan/outputs_pipeline_aligned/05_time_heterogeneity_models/coef_comparison/time_heterogeneity_variable_presence.csv

Fit comparison preview:


,run_name,time_window,hour_start,hour_end,row_query,model_name,y_col,y_base,y_transform,n_obs,...,cov_type,cluster_col,condition_number,dropped_constant_cols,dropped_alias_cols,auto_dropped_high_vif_cols,alias_keep_preference_model_cols,n_raw_x,n_model_x_before_dummies,n_interactions_created
0,h00_04,00:00-04:00,0,4,speed_kmh <= 40 and start_hour_local >= 0 and ...,ols_main,speed_kmh,speed_kmh,raw,12160,...,cluster,courier_id,2175.5473,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
1,h04_08,04:00-08:00,4,8,speed_kmh <= 40 and start_hour_local >= 4 and ...,ols_main,speed_kmh,speed_kmh,raw,10608,...,cluster,courier_id,2384.8061,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
2,h08_12,08:00-12:00,8,12,speed_kmh <= 40 and start_hour_local >= 8 and ...,ols_main,speed_kmh,speed_kmh,raw,89140,...,cluster,courier_id,2120.7268,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
3,h12_16,12:00-16:00,12,16,speed_kmh <= 40 and start_hour_local >= 12 and...,ols_main,speed_kmh,speed_kmh,raw,119020,...,cluster,courier_id,2135.5703,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
4,h16_20,16:00-20:00,16,20,speed_kmh <= 40 and start_hour_local >= 16 and...,ols_main,speed_kmh,speed_kmh,raw,149154,...,cluster,courier_id,2136.9032,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0
5,h20_24,20:00-24:00,20,24,speed_kmh <= 40 and start_hour_local >= 20 and...,ols_main,speed_kmh,speed_kmh,raw,71767,...,cluster,courier_id,2209.6590,[],[],[],"[""poi_count_restaurant_30m_lenw_mean"", ""restau...",16,16,0



Wide coefficient table preview:


,variable_group,variable,variable_base,00:00-04:00,04:00-08:00,08:00-12:00,12:00-16:00,16:00-20:00,20:00-24:00
0,constant,const,const,11.2527***,7.1953***,11.2633***,11.3314***,10.1436***,9.4727***
1,trip_state,onhand_order_count_start,onhand_order_count_start,0.5663***,0.4857***,0.2282***,0.3308***,0.2887***,0.4522***
2,trip_state,time_pressure_min,time_pressure_min,0.0123*,-0.0240**,-0.0020,0.0148***,0.0040**,0.0190***
3,trip_state,is_weekend_local,is_weekend_local,-0.0767,0.3847***,0.0393,-0.0462,0.1034***,0.0078
4,rider_behavior,rider_avg_orders_per_active_day,rider_avg_orders_per_active_day,0.0319**,0.0622***,0.0443***,0.0438***,0.0408***,0.0453***
5,rider_behavior,full_time,full_time,-0.0044,-0.1113,-0.0513,-0.0697,0.0037,-0.0457
6,rider_behavior,div1000_rider_mean_segment_distance_m,div1000_rider_mean_segment_distance_m,2.0980***,2.8366***,1.7598***,1.7347***,1.6886***,2.0634***
7,road_class_shares,road_class_share_levelThreeRoad_lenw_mean,road_class_share_levelThreeRoad_lenw_mean,0.9379***,0.1377,0.6707***,0.9842***,0.9298***,0.7622***
8,road_class_shares,road_class_share_secondaryRoad_lenw_mean,road_class_share_secondaryRoad_lenw_mean,3.2759***,1.6298***,1.3578***,1.6038***,1.5359***,1.6698***
9,road_class_shares,road_class_share_nationalRoad_lenw_mean,road_class_share_nationalRoad_lenw_mean,1.7406*,0.1471,1.4042***,1.1826***,1.4617***,2.0327***
